# Rebuild All Figures One PNG At A Time

Generated by `Figures/rebuild_figures.py`. Each markdown/code pair below corresponds to exactly one grouped PNG in `Figures/manifest.csv`. The code cell is self-contained for that PNG: it loads the relevant summarized CSVs or image inputs, defines the copied plotting block from the corresponding `make_figures.py` path when needed, saves directly into `Figures/`, and displays the result inline.

## 01. theory/01_theory_analytic/phi_by_analytic_solution_alpha0p1.png

Source image: `01_theory/01_theory_analytic/figures/phi_by_analytic_solution_alpha0p1.png`

Inputs:
- `01_theory/01_theory_analytic/summarized_outputs/phi_by_analytic_solution_alpha0p1.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/01_theory_analytic/phi_by_analytic_solution_alpha0p1.png'
input_paths = [ROOT / '01_theory/01_theory_analytic/summarized_outputs/phi_by_analytic_solution_alpha0p1.csv']

input_csv = input_paths[0]
df = pd.read_csv(input_csv).sort_values('r')
y_col = 'phi_rel' if 'phi_rel' in df.columns else 'phi'

output_png.parent.mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(7.0, 4.4))
plt.plot(df['r'], df[y_col], marker='o', linewidth=2.0, color='#2457a7')
plt.xlabel('d')
plt.ylabel('phi(d) - phi(d0)' if y_col == 'phi_rel' else 'phi(d)')
plt.title('Analytic full-RS solution, alpha=0.1')
plt.grid(True, alpha=0.28)
plt.tight_layout()
plt.savefig(output_png, dpi=180)
plt.show()
plt.close()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 02. theory/02_theory_sampling/logZ_split_distributions/N_160.png

Source image: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_160.png`

Inputs:
- `01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_160.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/02_theory_sampling/logZ_split_distributions/N_160.png'
input_paths = [ROOT / '01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_160.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _far_split_start(frame: pd.DataFrame) -> float | None:
    if "far_split_start_r" in frame.columns:
        values = pd.to_numeric(frame["far_split_start_r"], errors="coerce").dropna()
        if not values.empty:
            return float(values.iloc[0])
    if "payload_split" in frame.columns:
        far = frame.loc[frame["payload_split"].eq("far_split"), "r"]
        if not far.empty:
            return float(far.min())
    return None

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    labels = []
    for idx, value in enumerate(radii):
        labels.append(_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "")
    return labels

def _plot_value_column(frame: pd.DataFrame) -> tuple[str, str, bool]:
    if "signed_split_logZ_per_N_diff" in frame.columns:
        return "signed_split_logZ_per_N_diff", "signed split logZ diff per N", True
    return "split_logZ_per_N_diff", "absolute split logZ diff per N", False

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, split_threshold: float = 0.006) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    y_col, y_label, signed = _plot_value_column(frame)
    frame[y_col] = pd.to_numeric(frame[y_col], errors="coerce")
    frame = frame.dropna(subset=["r", y_col]).sort_values(["r"])
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    n_value = int(frame["N"].iloc[0]) if "N" in frame.columns else int(input_csv.stem.removeprefix("N_"))
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + n_value)
    if signed:
        color_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.98)), 1.0e-12)
        cmap = "coolwarm"
        vmin, vmax = -color_extent, color_extent
    else:
        vmax = max(float(frame[y_col].quantile(0.98)), 1.0e-12)
        cmap = "magma"
        vmin = 0.0
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    far_start = _far_split_start(frame)
    if far_start is not None:
        far_idx = next((idx for idx, radius in enumerate(radii) if np.isclose(radius, far_start)), None)
        if far_idx is not None:
            ax.axvline(far_idx - 0.5, color="#2f4b7c", linestyle="--", linewidth=1.2, alpha=0.8)
    if signed:
        y_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.995)), 1.0e-12) * 1.12
        ax.set_ylim(-y_extent, y_extent)
    else:
        ax.set_ylim(bottom=0.0)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel(y_label)
    ax.set_title(f"logZ split distributions, N={n_value}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

input_csv = input_paths[0]
plot_logz_split_distribution(input_csv, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 03. theory/02_theory_sampling/logZ_split_distributions/N_320.png

Source image: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_320.png`

Inputs:
- `01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_320.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/02_theory_sampling/logZ_split_distributions/N_320.png'
input_paths = [ROOT / '01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_320.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _far_split_start(frame: pd.DataFrame) -> float | None:
    if "far_split_start_r" in frame.columns:
        values = pd.to_numeric(frame["far_split_start_r"], errors="coerce").dropna()
        if not values.empty:
            return float(values.iloc[0])
    if "payload_split" in frame.columns:
        far = frame.loc[frame["payload_split"].eq("far_split"), "r"]
        if not far.empty:
            return float(far.min())
    return None

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    labels = []
    for idx, value in enumerate(radii):
        labels.append(_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "")
    return labels

def _plot_value_column(frame: pd.DataFrame) -> tuple[str, str, bool]:
    if "signed_split_logZ_per_N_diff" in frame.columns:
        return "signed_split_logZ_per_N_diff", "signed split logZ diff per N", True
    return "split_logZ_per_N_diff", "absolute split logZ diff per N", False

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, split_threshold: float = 0.006) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    y_col, y_label, signed = _plot_value_column(frame)
    frame[y_col] = pd.to_numeric(frame[y_col], errors="coerce")
    frame = frame.dropna(subset=["r", y_col]).sort_values(["r"])
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    n_value = int(frame["N"].iloc[0]) if "N" in frame.columns else int(input_csv.stem.removeprefix("N_"))
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + n_value)
    if signed:
        color_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.98)), 1.0e-12)
        cmap = "coolwarm"
        vmin, vmax = -color_extent, color_extent
    else:
        vmax = max(float(frame[y_col].quantile(0.98)), 1.0e-12)
        cmap = "magma"
        vmin = 0.0
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    far_start = _far_split_start(frame)
    if far_start is not None:
        far_idx = next((idx for idx, radius in enumerate(radii) if np.isclose(radius, far_start)), None)
        if far_idx is not None:
            ax.axvline(far_idx - 0.5, color="#2f4b7c", linestyle="--", linewidth=1.2, alpha=0.8)
    if signed:
        y_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.995)), 1.0e-12) * 1.12
        ax.set_ylim(-y_extent, y_extent)
    else:
        ax.set_ylim(bottom=0.0)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel(y_label)
    ax.set_title(f"logZ split distributions, N={n_value}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

input_csv = input_paths[0]
plot_logz_split_distribution(input_csv, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 04. theory/02_theory_sampling/logZ_split_distributions/N_40.png

Source image: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_40.png`

Inputs:
- `01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_40.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/02_theory_sampling/logZ_split_distributions/N_40.png'
input_paths = [ROOT / '01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_40.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _far_split_start(frame: pd.DataFrame) -> float | None:
    if "far_split_start_r" in frame.columns:
        values = pd.to_numeric(frame["far_split_start_r"], errors="coerce").dropna()
        if not values.empty:
            return float(values.iloc[0])
    if "payload_split" in frame.columns:
        far = frame.loc[frame["payload_split"].eq("far_split"), "r"]
        if not far.empty:
            return float(far.min())
    return None

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    labels = []
    for idx, value in enumerate(radii):
        labels.append(_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "")
    return labels

def _plot_value_column(frame: pd.DataFrame) -> tuple[str, str, bool]:
    if "signed_split_logZ_per_N_diff" in frame.columns:
        return "signed_split_logZ_per_N_diff", "signed split logZ diff per N", True
    return "split_logZ_per_N_diff", "absolute split logZ diff per N", False

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, split_threshold: float = 0.006) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    y_col, y_label, signed = _plot_value_column(frame)
    frame[y_col] = pd.to_numeric(frame[y_col], errors="coerce")
    frame = frame.dropna(subset=["r", y_col]).sort_values(["r"])
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    n_value = int(frame["N"].iloc[0]) if "N" in frame.columns else int(input_csv.stem.removeprefix("N_"))
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + n_value)
    if signed:
        color_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.98)), 1.0e-12)
        cmap = "coolwarm"
        vmin, vmax = -color_extent, color_extent
    else:
        vmax = max(float(frame[y_col].quantile(0.98)), 1.0e-12)
        cmap = "magma"
        vmin = 0.0
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    far_start = _far_split_start(frame)
    if far_start is not None:
        far_idx = next((idx for idx, radius in enumerate(radii) if np.isclose(radius, far_start)), None)
        if far_idx is not None:
            ax.axvline(far_idx - 0.5, color="#2f4b7c", linestyle="--", linewidth=1.2, alpha=0.8)
    if signed:
        y_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.995)), 1.0e-12) * 1.12
        ax.set_ylim(-y_extent, y_extent)
    else:
        ax.set_ylim(bottom=0.0)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel(y_label)
    ax.set_title(f"logZ split distributions, N={n_value}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

input_csv = input_paths[0]
plot_logz_split_distribution(input_csv, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 05. theory/02_theory_sampling/logZ_split_distributions/N_80.png

Source image: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_80.png`

Inputs:
- `01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_80.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/02_theory_sampling/logZ_split_distributions/N_80.png'
input_paths = [ROOT / '01_theory/02_theory_sampling/summarized_outputs/figure_inputs/logZ_split/N_80.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _far_split_start(frame: pd.DataFrame) -> float | None:
    if "far_split_start_r" in frame.columns:
        values = pd.to_numeric(frame["far_split_start_r"], errors="coerce").dropna()
        if not values.empty:
            return float(values.iloc[0])
    if "payload_split" in frame.columns:
        far = frame.loc[frame["payload_split"].eq("far_split"), "r"]
        if not far.empty:
            return float(far.min())
    return None

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    labels = []
    for idx, value in enumerate(radii):
        labels.append(_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "")
    return labels

def _plot_value_column(frame: pd.DataFrame) -> tuple[str, str, bool]:
    if "signed_split_logZ_per_N_diff" in frame.columns:
        return "signed_split_logZ_per_N_diff", "signed split logZ diff per N", True
    return "split_logZ_per_N_diff", "absolute split logZ diff per N", False

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, split_threshold: float = 0.006) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    y_col, y_label, signed = _plot_value_column(frame)
    frame[y_col] = pd.to_numeric(frame[y_col], errors="coerce")
    frame = frame.dropna(subset=["r", y_col]).sort_values(["r"])
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    n_value = int(frame["N"].iloc[0]) if "N" in frame.columns else int(input_csv.stem.removeprefix("N_"))
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + n_value)
    if signed:
        color_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.98)), 1.0e-12)
        cmap = "coolwarm"
        vmin, vmax = -color_extent, color_extent
    else:
        vmax = max(float(frame[y_col].quantile(0.98)), 1.0e-12)
        cmap = "magma"
        vmin = 0.0
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), y_col].to_numpy(float)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    far_start = _far_split_start(frame)
    if far_start is not None:
        far_idx = next((idx for idx, radius in enumerate(radii) if np.isclose(radius, far_start)), None)
        if far_idx is not None:
            ax.axvline(far_idx - 0.5, color="#2f4b7c", linestyle="--", linewidth=1.2, alpha=0.8)
    if signed:
        y_extent = max(float(np.nanquantile(np.abs(frame[y_col]), 0.995)), 1.0e-12) * 1.12
        ax.set_ylim(-y_extent, y_extent)
    else:
        ax.set_ylim(bottom=0.0)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel(y_label)
    ax.set_title(f"logZ split distributions, N={n_value}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

input_csv = input_paths[0]
plot_logz_split_distribution(input_csv, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 06. theory/02_theory_sampling/phi_by_sampling/phi_by_sampling.png

Source image: `01_theory/02_theory_sampling/figures/phi_by_sampling/phi_by_sampling.png`

Inputs:
- `01_theory/02_theory_sampling/summarized_outputs/figure_inputs/phi_by_sampling`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/02_theory_sampling/phi_by_sampling/phi_by_sampling.png'
input_paths = [ROOT / '01_theory/02_theory_sampling/summarized_outputs/figure_inputs/phi_by_sampling']

input_root = input_paths[0]
files = sorted(input_root.glob('N_*.csv'))
if not files:
    raise FileNotFoundError(f'no N_*.csv files found under {input_root}')
df = pd.concat((pd.read_csv(path) for path in files), ignore_index=True, sort=False).sort_values(['N', 'r'])

def series_value(frame: pd.DataFrame) -> pd.Series:
    if 'phi_emp_rel' in frame.columns:
        return frame['phi_emp_rel']
    base = frame.sort_values('r')['phi_emp'].iloc[0]
    return frame['phi_emp'] - base

output_png.parent.mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(7.4, 4.8))
for n_value, group in df.groupby('N', sort=True):
    group = group.sort_values('r')
    plt.plot(group['r'], series_value(group), marker='o', linewidth=1.7, label=f'N={int(n_value)}')
plt.xlabel('d')
plt.ylabel('empirical phi(d) - phi(d0)')
plt.title('Two-pool shell sampling, alpha=0.1')
plt.grid(True, alpha=0.28)
plt.legend(title='system size', fontsize=8)
plt.tight_layout()
plt.savefig(output_png, dpi=180)
plt.show()
plt.close()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 07. theory/summary/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png

Source image: `01_theory/figures/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png`

Inputs:
- `01_theory/01_theory_analytic/summarized_outputs/phi_by_analytic_solution_alpha0p1.csv`
- `01_theory/02_theory_sampling/summarized_outputs/figure_inputs/phi_by_sampling`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'theory/summary/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png'
input_paths = [ROOT / '01_theory/01_theory_analytic/summarized_outputs/phi_by_analytic_solution_alpha0p1.csv', ROOT / '01_theory/02_theory_sampling/summarized_outputs/figure_inputs/phi_by_sampling']

analytic_csv, sampling_root = input_paths
analytic = pd.read_csv(analytic_csv).sort_values('r')
sampling_files = sorted(sampling_root.glob('N_*.csv'))
if not sampling_files:
    raise FileNotFoundError(f'no N_*.csv files found under {sampling_root}')
sampling = pd.concat((pd.read_csv(path) for path in sampling_files), ignore_index=True, sort=False).sort_values(['N', 'r'])

def analytic_relative_phi(frame: pd.DataFrame) -> pd.Series:
    if 'phi_rel' in frame.columns:
        return frame['phi_rel']
    ordered = frame.sort_values('r')
    return ordered['phi'] - ordered['phi'].iloc[0]

def sampling_relative_phi(frame: pd.DataFrame) -> pd.Series:
    if 'phi_emp_rel' in frame.columns:
        return frame['phi_emp_rel']
    ordered = frame.sort_values('r')
    return ordered['phi_emp'] - ordered['phi_emp'].iloc[0]

plt.figure(figsize=(8.5, 5.1))
plt.plot(analytic['r'], analytic_relative_phi(analytic), color='black', linewidth=2.4, label='analytic full-RS')
for n_value, group in sampling.groupby('N', sort=True):
    group = group.sort_values('r')
    plt.plot(group['r'], sampling_relative_phi(group), marker='o', markersize=3.2, linewidth=1.55, label=f'N={int(n_value)} sampling')
plt.xlabel('d')
plt.ylabel('phi(d) - phi(d0)')
plt.title('Analytic vs shell sampling, alpha=0.1')
plt.grid(True, alpha=0.28)
plt.legend(fontsize=8)
plt.tight_layout()
output_png.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_png, dpi=300)
plt.show()
plt.close()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 08. dnn_synthetic/01_dataset/sample_figure.png

Source image: `02_dnn_synthetic/figures/01_dataset/sample_figure.png`

Inputs:
- `02_dnn_synthetic/01_dataset/summarized_outputs/figure_inputs/sample_figures/selected_sample_indices.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/01_dataset/sample_figure.png'
input_paths = [ROOT / '02_dnn_synthetic/01_dataset/summarized_outputs/figure_inputs/sample_figures/selected_sample_indices.csv']

sample_csv = input_paths[0]
sample_frame = pd.read_csv(sample_csv)
if sample_frame.empty:
    raise ValueError(f'{sample_csv} is empty')

ncols = 6
nrows = int(math.ceil(len(sample_frame) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.7 * ncols, 3.45 * nrows), squeeze=False)
for ax in axes.ravel():
    ax.axis('off')

for idx, row in sample_frame.iterrows():
    image_path = ROOT / str(row['source_image_path'])
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    ax = axes[int(idx) // ncols, int(idx) % ncols]
    ax.imshow(mpimg.imread(image_path))
    ax.axis('off')

fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.03, hspace=0.05)
output_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_png, dpi=220, bbox_inches='tight', pad_inches=0.04)
plt.show()
plt.close(fig)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 09. dnn_synthetic/01_dataset/spin_dynamics_phase_transition.png

Source image: `02_dnn_synthetic/figures/01_dataset/spin_dynamics_phase_transition.png`

Inputs:
- `02_dnn_synthetic/01_dataset/summarized_outputs/figure_inputs/spin_dynamics/spin_alignment_by_beta.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/01_dataset/spin_dynamics_phase_transition.png'
input_paths = [ROOT / '02_dnn_synthetic/01_dataset/summarized_outputs/figure_inputs/spin_dynamics/spin_alignment_by_beta.csv']

spin_frame = pd.read_csv(input_paths[0])
beta = pd.to_numeric(spin_frame['beta_ising'], errors='coerce').to_numpy(dtype=float)
mean = pd.to_numeric(spin_frame['mean_edge_alignment'], errors='coerce').to_numpy(dtype=float)
sem = pd.to_numeric(spin_frame['sem_edge_alignment'], errors='coerce').to_numpy(dtype=float)
mask = np.isfinite(beta) & np.isfinite(mean) & np.isfinite(sem)
beta = beta[mask]
mean = mean[mask]
sem = sem[mask]
order = np.argsort(beta)

fig, ax = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
ax.plot(beta[order], mean[order], color='#252525', linewidth=2.2, marker='o', markersize=4.8)
ax.fill_between(beta[order], mean[order] - sem[order], mean[order] + sem[order], color='#5b8db8', alpha=0.22, linewidth=0.0)
ax.set_xlabel('inverse temperature beta (lower T to the right)')
ax.set_ylabel('mean edge spin alignment <s_i s_j>')
ax.set_title('Spin-dynamics snapshots show temperature-driven ordering')
ax.set_ylim(0.0, 0.96)
ax.set_xlim(float(beta.min()) - 0.01, float(beta.max()) + 0.01)
ax.grid(True, color='#d9d9d9', linewidth=0.8, alpha=0.75)
ax.text(0.03, 0.93, '90 final snapshots per beta\n2000 Kawasaki sweeps per snapshot', transform=ax.transAxes, ha='left', va='top', fontsize=9, bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white', 'edgecolor': '#bbbbbb', 'alpha': 0.92})
top_ax = ax.twiny()
top_ax.set_xlim(ax.get_xlim())
top_ticks = np.asarray([0.05, 0.10, 0.20, 0.30, 0.39], dtype=float)
top_ax.set_xticks(top_ticks)
top_ax.set_xticklabels([f'{1.0 / tick:.1f}' for tick in top_ticks])
top_ax.set_xlabel('temperature T = 1 / beta')
output_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_png, dpi=220, bbox_inches='tight')
plt.show()
plt.close(fig)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 10. dnn_synthetic/02_complexity_measure/beta_complexity_figure.png

Source image: `02_dnn_synthetic/figures/02_complexity_measure/beta_complexity_figure.png`

Inputs:
- `02_dnn_synthetic/02_complexity_measure/summarized_outputs/beta_complexity_summary.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/02_complexity_measure/beta_complexity_figure.png'
input_paths = [ROOT / '02_dnn_synthetic/02_complexity_measure/summarized_outputs/beta_complexity_summary.csv']

frame = pd.read_csv(input_paths[0])
beta = pd.to_numeric(frame['beta'], errors='coerce').to_numpy(dtype=float)
mean = pd.to_numeric(frame['complexity_mean'], errors='coerce').to_numpy(dtype=float)
se = pd.to_numeric(frame['complexity_se'], errors='coerce').to_numpy(dtype=float)
mask = np.isfinite(beta) & np.isfinite(mean) & np.isfinite(se)
beta = beta[mask]
mean = mean[mask]
se = se[mask]
order = np.argsort(beta)
r = float(np.corrcoef(beta[order], mean[order])[0, 1]) if len(beta) > 1 else float('nan')

fig, ax = plt.subplots(figsize=(6.8, 4.4), constrained_layout=True)
ax.errorbar(beta[order], mean[order], yerr=se[order], fmt='o-', color='#284f8f', ecolor='#8aa7d6', capsize=3)
ax.set_xlabel(r'$\beta$')
ax.set_ylabel('3-NN label-disagreement complexity')
ax.set_title(f'Beta vs complexity (Pearson r={r:.3f})')
ax.grid(True, alpha=0.25)
output_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_png, dpi=240)
plt.show()
plt.close(fig)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 11. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p05.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p05.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p05.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p05.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p05.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 12. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p07.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p07.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p07.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p07.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p07.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 13. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p09.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p09.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p09.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p09.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p09.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 14. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p11.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p11.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p11.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p11.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p11.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 15. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p13.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p13.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p13.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p13.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p13.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 16. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p15.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p15.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p15.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p15.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p15.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 17. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p17.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p17.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p17.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p17.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p17.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 18. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p19.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p19.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p19.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p19.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p19.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 19. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p21.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p21.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p21.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p21.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p21.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 20. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p23.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p23.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p23.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p23.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p23.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 21. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p25.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p25.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p25.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p25.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p25.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 22. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p27.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p27.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p27.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p27.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p27.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 23. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p29.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p29.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p29.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p29.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p29.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 24. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p31.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p31.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p31.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p31.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p31.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 25. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p33.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p33.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p33.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p33.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p33.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 26. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p35.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p35.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p35.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p35.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p35.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 27. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p37.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p37.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p37.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p37.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p37.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 28. dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p39.png

Source image: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/beta_0p39.png`

Inputs:
- `02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p39.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/04_sampling/logZ_split_distributions/beta_0p39.png'
input_paths = [ROOT / '02_dnn_synthetic/04_sampling/summarized_outputs/figure_inputs/logZ_split/beta_0p39.csv']

def radius_label(value: float) -> str:
    return f"{value:g}"

def xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(input_csv: Path, output_png: Path, *, max_scatter_per_radius: int) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    beta = float(frame["beta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(beta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, beta={beta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 29. dnn_synthetic/05_proxy_local_entropy/derivative_phi_d_curve.png

Source image: `02_dnn_synthetic/figures/05_proxy_local_entropy/derivative_phi_d_curve.png`

Inputs:
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/05_proxy_local_entropy/derivative_phi_d_curve.png'
input_paths = [ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv']

def group_curves(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    required = {"beta", "radius", value_key}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"missing columns for curve plot: {', '.join(missing)}")

    work = frame.copy()
    for col in ("beta", "radius", value_key):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if sem_key is not None and sem_key in work.columns:
        work[sem_key] = pd.to_numeric(work[sem_key], errors="coerce")

    grouped: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for beta, group in work.dropna(subset=["beta", "radius", value_key]).groupby("beta", sort=True):
        group = group.sort_values("radius")
        radius = group["radius"].to_numpy(dtype=float)
        value = group[value_key].to_numpy(dtype=float)
        sem: np.ndarray | None = None
        if sem_key is not None and sem_key in group.columns:
            sem_values = group[sem_key].to_numpy(dtype=float)
            if np.isfinite(sem_values).any():
                sem = sem_values
        grouped[round(float(beta), 8)] = (radius, value, sem)
    return grouped

def plot_curve_frame(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
    ylabel: str,
    title: str,
    path: Path,
    *,
    figsize: tuple[float, float] = (7.4, 4.8),
    dpi: int = 240,
    xscale: str = "linear",
    colorbar: bool = True,
    legend: bool = False,
) -> None:
    curves = group_curves(frame, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for beta, (radius, value, sem) in curves.items():
        color = cmap(norm(beta))
        label = rf"$\beta={beta:.2f}$"
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color, label=label)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    if colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label(r"$\beta$")
    if legend:
        ax.legend(frameon=False, fontsize=7.0, ncol=2)

    ax.set_xscale(xscale)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    plt.close(fig)

value_key = 'dphi_full_dr_mean'
sem_key = 'dphi_full_dr_sem'
ylabel = '$d\\phi/dd$'
title = 'Synthetic derivative of $\\phi(d)$'
frame = pd.read_csv(input_paths[0])
plot_curve_frame(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 30. dnn_synthetic/05_proxy_local_entropy/derivative_phi_energetic_d_curve.png

Source image: `02_dnn_synthetic/figures/05_proxy_local_entropy/derivative_phi_energetic_d_curve.png`

Inputs:
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/05_proxy_local_entropy/derivative_phi_energetic_d_curve.png'
input_paths = [ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv']

def group_curves(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    required = {"beta", "radius", value_key}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"missing columns for curve plot: {', '.join(missing)}")

    work = frame.copy()
    for col in ("beta", "radius", value_key):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if sem_key is not None and sem_key in work.columns:
        work[sem_key] = pd.to_numeric(work[sem_key], errors="coerce")

    grouped: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for beta, group in work.dropna(subset=["beta", "radius", value_key]).groupby("beta", sort=True):
        group = group.sort_values("radius")
        radius = group["radius"].to_numpy(dtype=float)
        value = group[value_key].to_numpy(dtype=float)
        sem: np.ndarray | None = None
        if sem_key is not None and sem_key in group.columns:
            sem_values = group[sem_key].to_numpy(dtype=float)
            if np.isfinite(sem_values).any():
                sem = sem_values
        grouped[round(float(beta), 8)] = (radius, value, sem)
    return grouped

def plot_curve_frame(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
    ylabel: str,
    title: str,
    path: Path,
    *,
    figsize: tuple[float, float] = (7.4, 4.8),
    dpi: int = 240,
    xscale: str = "linear",
    colorbar: bool = True,
    legend: bool = False,
) -> None:
    curves = group_curves(frame, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for beta, (radius, value, sem) in curves.items():
        color = cmap(norm(beta))
        label = rf"$\beta={beta:.2f}$"
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color, label=label)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    if colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label(r"$\beta$")
    if legend:
        ax.legend(frameon=False, fontsize=7.0, ncol=2)

    ax.set_xscale(xscale)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    plt.close(fig)

value_key = 'dphi_energy_dr_mean'
sem_key = 'dphi_energy_dr_sem'
ylabel = 'energetic $d\\phi/dd$'
title = 'Synthetic energetic derivative of $\\phi(d)$'
frame = pd.read_csv(input_paths[0])
plot_curve_frame(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 31. dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_beta.png

Source image: `02_dnn_synthetic/figures/05_proxy_local_entropy/phase_like_A_by_beta.png`

Inputs:
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_beta/phase_like_A_by_beta.csv`
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_beta/phase_derivative_curves.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_beta.png'
input_paths = [ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_beta/phase_like_A_by_beta.csv', ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_beta/phase_derivative_curves.csv']

def group_curves(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    required = {"beta", "radius", value_key}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"missing columns for curve plot: {', '.join(missing)}")

    work = frame.copy()
    for col in ("beta", "radius", value_key):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if sem_key is not None and sem_key in work.columns:
        work[sem_key] = pd.to_numeric(work[sem_key], errors="coerce")

    grouped: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for beta, group in work.dropna(subset=["beta", "radius", value_key]).groupby("beta", sort=True):
        group = group.sort_values("radius")
        radius = group["radius"].to_numpy(dtype=float)
        value = group[value_key].to_numpy(dtype=float)
        sem: np.ndarray | None = None
        if sem_key is not None and sem_key in group.columns:
            sem_values = group[sem_key].to_numpy(dtype=float)
            if np.isfinite(sem_values).any():
                sem = sem_values
        grouped[round(float(beta), 8)] = (radius, value, sem)
    return grouped

def plot_phase_panel(
    phase_frame: pd.DataFrame,
    derivative_frame: pd.DataFrame,
    x_key: str,
    x_label: str,
    title: str,
    output_path: Path,
) -> None:
    curve_groups = group_curves(derivative_frame, "dphi_dr_smooth_mean", "dphi_dr_smooth_sem")
    if not curve_groups:
        raise ValueError(f"no finite derivative rows to plot for {output_path}")

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curve_groups), max(curve_groups))
    has_band = False
    for beta, (radius, value, sem) in curve_groups.items():
        color = cmap(norm(beta))
        ax_left.plot(radius, value, linewidth=1.1, alpha=0.85, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax_left.fill_between(radius, value - err, value + err, color=color, alpha=0.09, linewidth=0)
            has_band = True

    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel(r"energetic $d\phi/dd$")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.22)
    if has_band:
        ax_left.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax_left.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    x = pd.to_numeric(phase_frame[x_key], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(phase_frame["A_transition_total_variation_mean"], errors="coerce").to_numpy(dtype=float)
    yerr = pd.to_numeric(phase_frame.get("A_transition_total_variation_sem"), errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    yerr = yerr[mask]
    order = np.argsort(x)
    if np.isfinite(yerr).any():
        ax_right.errorbar(
            x[order],
            y[order],
            yerr=np.nan_to_num(yerr[order], nan=0.0),
            marker="o",
            linewidth=1.5,
            capsize=2.5,
            color="#7a3e9d",
        )
    else:
        ax_right.plot(x[order], y[order], "o-", color="#7a3e9d", linewidth=1.4, markersize=4.5)
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title(title)
    ax_right.grid(True, alpha=0.25)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=240)
    plt.close(fig)

x_key = 'beta'
x_label = '$\\beta$'
title = 'A measure by beta'
phase_frame = pd.read_csv(input_paths[0])
derivative_frame = pd.read_csv(input_paths[1])
plot_phase_panel(phase_frame, derivative_frame, x_key, x_label, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 32. dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_complexity.png

Source image: `02_dnn_synthetic/figures/05_proxy_local_entropy/phase_like_A_by_complexity.png`

Inputs:
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv`
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_complexity.png'
input_paths = [ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv']

def group_curves(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    required = {"beta", "radius", value_key}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"missing columns for curve plot: {', '.join(missing)}")

    work = frame.copy()
    for col in ("beta", "radius", value_key):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if sem_key is not None and sem_key in work.columns:
        work[sem_key] = pd.to_numeric(work[sem_key], errors="coerce")

    grouped: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for beta, group in work.dropna(subset=["beta", "radius", value_key]).groupby("beta", sort=True):
        group = group.sort_values("radius")
        radius = group["radius"].to_numpy(dtype=float)
        value = group[value_key].to_numpy(dtype=float)
        sem: np.ndarray | None = None
        if sem_key is not None and sem_key in group.columns:
            sem_values = group[sem_key].to_numpy(dtype=float)
            if np.isfinite(sem_values).any():
                sem = sem_values
        grouped[round(float(beta), 8)] = (radius, value, sem)
    return grouped

def plot_phase_panel(
    phase_frame: pd.DataFrame,
    derivative_frame: pd.DataFrame,
    x_key: str,
    x_label: str,
    title: str,
    output_path: Path,
) -> None:
    curve_groups = group_curves(derivative_frame, "dphi_dr_smooth_mean", "dphi_dr_smooth_sem")
    if not curve_groups:
        raise ValueError(f"no finite derivative rows to plot for {output_path}")

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curve_groups), max(curve_groups))
    has_band = False
    for beta, (radius, value, sem) in curve_groups.items():
        color = cmap(norm(beta))
        ax_left.plot(radius, value, linewidth=1.1, alpha=0.85, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax_left.fill_between(radius, value - err, value + err, color=color, alpha=0.09, linewidth=0)
            has_band = True

    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel(r"energetic $d\phi/dd$")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.22)
    if has_band:
        ax_left.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax_left.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    x = pd.to_numeric(phase_frame[x_key], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(phase_frame["A_transition_total_variation_mean"], errors="coerce").to_numpy(dtype=float)
    yerr = pd.to_numeric(phase_frame.get("A_transition_total_variation_sem"), errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    yerr = yerr[mask]
    order = np.argsort(x)
    if np.isfinite(yerr).any():
        ax_right.errorbar(
            x[order],
            y[order],
            yerr=np.nan_to_num(yerr[order], nan=0.0),
            marker="o",
            linewidth=1.5,
            capsize=2.5,
            color="#7a3e9d",
        )
    else:
        ax_right.plot(x[order], y[order], "o-", color="#7a3e9d", linewidth=1.4, markersize=4.5)
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title(title)
    ax_right.grid(True, alpha=0.25)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=240)
    plt.close(fig)

x_key = 'complexity_mean'
x_label = '3-NN complexity'
title = 'A measure by complexity'
phase_frame = pd.read_csv(input_paths[0])
derivative_frame = pd.read_csv(input_paths[1])
plot_phase_panel(phase_frame, derivative_frame, x_key, x_label, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 33. dnn_synthetic/05_proxy_local_entropy/phi_d_curve.png

Source image: `02_dnn_synthetic/figures/05_proxy_local_entropy/phi_d_curve.png`

Inputs:
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/05_proxy_local_entropy/phi_d_curve.png'
input_paths = [ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv']

def group_curves(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    required = {"beta", "radius", value_key}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"missing columns for curve plot: {', '.join(missing)}")

    work = frame.copy()
    for col in ("beta", "radius", value_key):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if sem_key is not None and sem_key in work.columns:
        work[sem_key] = pd.to_numeric(work[sem_key], errors="coerce")

    grouped: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for beta, group in work.dropna(subset=["beta", "radius", value_key]).groupby("beta", sort=True):
        group = group.sort_values("radius")
        radius = group["radius"].to_numpy(dtype=float)
        value = group[value_key].to_numpy(dtype=float)
        sem: np.ndarray | None = None
        if sem_key is not None and sem_key in group.columns:
            sem_values = group[sem_key].to_numpy(dtype=float)
            if np.isfinite(sem_values).any():
                sem = sem_values
        grouped[round(float(beta), 8)] = (radius, value, sem)
    return grouped

def plot_curve_frame(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
    ylabel: str,
    title: str,
    path: Path,
    *,
    figsize: tuple[float, float] = (7.4, 4.8),
    dpi: int = 240,
    xscale: str = "linear",
    colorbar: bool = True,
    legend: bool = False,
) -> None:
    curves = group_curves(frame, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for beta, (radius, value, sem) in curves.items():
        color = cmap(norm(beta))
        label = rf"$\beta={beta:.2f}$"
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color, label=label)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    if colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label(r"$\beta$")
    if legend:
        ax.legend(frameon=False, fontsize=7.0, ncol=2)

    ax.set_xscale(xscale)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    plt.close(fig)

value_key = 'phi_full_mean'
sem_key = 'phi_full_sem'
ylabel = '$\\phi(d)$'
title = 'Synthetic $\\phi(d)$ by distance'
frame = pd.read_csv(input_paths[0])
plot_curve_frame(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 34. dnn_synthetic/05_proxy_local_entropy/phi_energetic_d_curve.png

Source image: `02_dnn_synthetic/figures/05_proxy_local_entropy/phi_energetic_d_curve.png`

Inputs:
- `02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_synthetic/05_proxy_local_entropy/phi_energetic_d_curve.png'
input_paths = [ROOT / '02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv']

def group_curves(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    required = {"beta", "radius", value_key}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"missing columns for curve plot: {', '.join(missing)}")

    work = frame.copy()
    for col in ("beta", "radius", value_key):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if sem_key is not None and sem_key in work.columns:
        work[sem_key] = pd.to_numeric(work[sem_key], errors="coerce")

    grouped: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for beta, group in work.dropna(subset=["beta", "radius", value_key]).groupby("beta", sort=True):
        group = group.sort_values("radius")
        radius = group["radius"].to_numpy(dtype=float)
        value = group[value_key].to_numpy(dtype=float)
        sem: np.ndarray | None = None
        if sem_key is not None and sem_key in group.columns:
            sem_values = group[sem_key].to_numpy(dtype=float)
            if np.isfinite(sem_values).any():
                sem = sem_values
        grouped[round(float(beta), 8)] = (radius, value, sem)
    return grouped

def plot_curve_frame(
    frame: pd.DataFrame,
    value_key: str,
    sem_key: str | None,
    ylabel: str,
    title: str,
    path: Path,
    *,
    figsize: tuple[float, float] = (7.4, 4.8),
    dpi: int = 240,
    xscale: str = "linear",
    colorbar: bool = True,
    legend: bool = False,
) -> None:
    curves = group_curves(frame, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for beta, (radius, value, sem) in curves.items():
        color = cmap(norm(beta))
        label = rf"$\beta={beta:.2f}$"
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color, label=label)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    if colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label(r"$\beta$")
    if legend:
        ax.legend(frameon=False, fontsize=7.0, ncol=2)

    ax.set_xscale(xscale)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    plt.close(fig)

value_key = 'phi_energy_mean'
sem_key = 'phi_energy_sem'
ylabel = 'energetic $\\phi(d)$'
title = 'Synthetic energetic $\\phi(d)$ by distance'
frame = pd.read_csv(input_paths[0])
plot_curve_frame(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 35. dnn_mnist/01_dataset/label_noise_sweep/sample_figure.png

Source image: `03_dnn_mnist/figures/01_dataset/label_noise_sweep/sample_figure.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/01_dataset/summarized_outputs/figure_inputs/sample_figure/selected_sample_indices.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/01_dataset/label_noise_sweep/sample_figure.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/01_dataset/summarized_outputs/figure_inputs/sample_figure/selected_sample_indices.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
FIGURE_ROOT = output_png.parent
SAMPLE_FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/01_dataset/summarized_outputs/figure_inputs/sample_figure'
SAMPLE_FIGURE_PATH = output_png

def repo_path(relative_path: str) -> Path:
    path = Path(relative_path)
    if path.is_absolute():
        return path
    candidates = [DNN_ROOT / path]
    text = path.as_posix()
    marker = '03_dnn_mnist/'
    if marker in text:
        candidates.append(DNN_ROOT / text.split(marker, 1)[1])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

def load_csv_rows(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def _resolve_source(path_value: str) -> Path:
    return repo_path(path_value)

def build_sample_figure() -> None:
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
    rows = load_csv_rows(SAMPLE_FIGURE_INPUT_ROOT / "selected_sample_indices.csv")
    if not rows:
        raise ValueError("sample figure summary is empty")

    rows_by_eta: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        rows_by_eta[row["noise_eta"]].append(row)

    eta_names = sorted(rows_by_eta, key=lambda name: float(name.replace("noise_eta_", "").replace("p", ".")))
    ncols = max(len(rows_by_eta[name]) for name in eta_names)
    nrows = len(eta_names)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(1.25 * ncols, 1.45 * nrows),
        squeeze=False,
        constrained_layout=True,
    )

    for ax in axes.ravel():
        ax.axis("off")

    payload_cache: dict[Path, np.lib.npyio.NpzFile] = {}
    try:
        for row_idx, eta_name in enumerate(eta_names):
            eta_rows = sorted(rows_by_eta[eta_name], key=lambda row: int(row["sample_order"]))
            for col_idx, row in enumerate(eta_rows):
                dataset = _resolve_source(row["source_dataset_path"])
                if dataset not in payload_cache:
                    payload_cache[dataset] = np.load(dataset)
                data = payload_cache[dataset]
                sample_index = int(row["sample_index"])
                image = data[row["source_array"]][sample_index].reshape(10, 10)

                ax = axes[row_idx, col_idx]
                ax.imshow(image, cmap="gray", vmin=0, vmax=255)
                ax.set_xticks([])
                ax.set_yticks([])
                marker = "*" if int(row["flipped"]) else ""
                ax.set_title(f"d={row['digit']} y={row['label']}{marker}", fontsize=7)
                if col_idx == 0:
                    ax.text(
                        0.03,
                        0.96,
                        f"eta={row['eta']}",
                        transform=ax.transAxes,
                        ha="left",
                        va="top",
                        color="white",
                        fontsize=7,
                        bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55, "pad": 1.5},
                    )
        fig.savefig(SAMPLE_FIGURE_PATH, dpi=220, bbox_inches="tight", pad_inches=0.03)
    finally:
        for data in payload_cache.values():
            data.close()
        plt.close(fig)

build_sample_figure()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 36. dnn_mnist/01_dataset/manual_rules/sample_figure.png

Source image: `03_dnn_mnist/figures/01_dataset/manual_rules/sample_figure.png`

Inputs:
- `03_dnn_mnist/manual_rules/01_dataset/summarized_outputs/figure_inputs/sample_figure/selected_sample_indices.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/01_dataset/manual_rules/sample_figure.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/01_dataset/summarized_outputs/figure_inputs/sample_figure/selected_sample_indices.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
FIGURE_ROOT = output_png.parent
SAMPLE_FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/01_dataset/summarized_outputs/figure_inputs/sample_figure'
SAMPLE_FIGURE_PATH = output_png

def repo_path(relative_path: str) -> Path:
    path = Path(relative_path)
    if path.is_absolute():
        return path
    candidates = [DNN_ROOT / path]
    text = path.as_posix()
    marker = '03_dnn_mnist/'
    if marker in text:
        candidates.append(DNN_ROOT / text.split(marker, 1)[1])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

def load_csv_rows(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def _resolve_source(path_value: str) -> Path:
    return repo_path(path_value)

def build_sample_figure() -> None:
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
    rows = load_csv_rows(SAMPLE_FIGURE_INPUT_ROOT / "selected_sample_indices.csv")
    if not rows:
        raise ValueError("sample figure summary is empty")

    rows_by_rule: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        rows_by_rule[row["rule_id"]].append(row)

    rule_ids = sorted(rows_by_rule)
    ncols = max(len(rows_by_rule[rule_id]) for rule_id in rule_ids)
    nrows = len(rule_ids)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(1.25 * ncols, 1.45 * nrows),
        squeeze=False,
        constrained_layout=True,
    )

    for ax in axes.ravel():
        ax.axis("off")

    payload_cache: dict[Path, np.lib.npyio.NpzFile] = {}
    try:
        for row_idx, rule_id in enumerate(rule_ids):
            rule_rows = sorted(rows_by_rule[rule_id], key=lambda row: int(row["sample_order"]))
            for col_idx, row in enumerate(rule_rows):
                dataset = _resolve_source(row["source_dataset_path"])
                if dataset not in payload_cache:
                    payload_cache[dataset] = np.load(dataset)
                data = payload_cache[dataset]
                sample_index = int(row["sample_index"])
                image = data[row["source_array"]][sample_index].reshape(10, 10)

                ax = axes[row_idx, col_idx]
                ax.imshow(image, cmap="gray", vmin=0, vmax=255)
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_title(f"d={row['digit']} y={row['sample_label']}", fontsize=7)
                if col_idx == 0:
                    ax.text(
                        0.03,
                        0.96,
                        row["rule_label"],
                        transform=ax.transAxes,
                        ha="left",
                        va="top",
                        color="white",
                        fontsize=7,
                        bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55, "pad": 1.5},
                    )
        fig.savefig(SAMPLE_FIGURE_PATH, dpi=220, bbox_inches="tight", pad_inches=0.03)
    finally:
        for data in payload_cache.values():
            data.close()
        plt.close(fig)

build_sample_figure()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 37. dnn_mnist/02_complexity_measure/label_noise_sweep/eta_complexity_figure.png

Source image: `03_dnn_mnist/figures/02_complexity_measure/label_noise_sweep/eta_complexity_figure.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/02_complexity_measure/label_noise_sweep/eta_complexity_figure.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv']

SUMMARY_PATH = input_paths[0]
FIGURE_PATH = output_png

def _read_summary(path: Path) -> list[dict[str, float]]:
    if not path.exists():
        raise FileNotFoundError(f"run src/make_summarized_outputs.py first: {path}")

    required_fields = ("eta", "complexity_mean", "complexity_se")
    rows: list[dict[str, float]] = []
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []
        missing = [field for field in required_fields if field not in fieldnames]
        if missing:
            raise ValueError(f"{path} is missing columns: {', '.join(missing)}")
        for row in reader:
            rows.append(
                {
                    "eta": float(row["eta"]),
                    "complexity_mean": float(row["complexity_mean"]),
                    "complexity_se": float(row["complexity_se"]),
                }
            )

    if not rows:
        raise ValueError(f"no rows for {path}")
    return rows

def _plot(summary_rows: list[dict[str, float]]) -> None:
    eta = np.asarray([float(row["eta"]) for row in summary_rows], dtype=np.float64)
    mean = np.asarray([float(row["complexity_mean"]) for row in summary_rows], dtype=np.float64)
    se = np.asarray([float(row["complexity_se"]) for row in summary_rows], dtype=np.float64)
    r, p_value = pearsonr(eta, mean)

    FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6.8, 4.4), constrained_layout=True)
    ax.errorbar(eta, mean, yerr=se, fmt="o-", color="#284f8f", ecolor="#8aa7d6", capsize=3)
    ax.set_xlabel("label noise eta")
    ax.set_ylabel("3-NN label-disagreement complexity")
    ax.set_title(f"Eta vs complexity (Pearson r={r:.3f}, p={p_value:.1e})")
    ax.grid(True, alpha=0.25)
    fig.savefig(FIGURE_PATH, dpi=240)
    plt.close(fig)

summary_rows = _read_summary(SUMMARY_PATH)
_plot(summary_rows)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 38. dnn_mnist/02_complexity_measure/manual_rules/manual_rule_complexity_figure.png

Source image: `03_dnn_mnist/figures/02_complexity_measure/manual_rules/manual_rule_complexity_figure.png`

Inputs:
- `03_dnn_mnist/manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/02_complexity_measure/manual_rules/manual_rule_complexity_figure.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv']

SUMMARY_PATH = input_paths[0]
FIGURE_PATH = output_png

def _read_summary(path: Path) -> list[dict[str, float | str]]:
    if not path.exists():
        raise FileNotFoundError(f"run src/make_summarized_outputs.py first: {path}")

    required_fields = ("label", "rule_order", "complexity_mean", "complexity_se")
    rows: list[dict[str, float | str]] = []
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []
        missing = [field for field in required_fields if field not in fieldnames]
        if missing:
            raise ValueError(f"{path} is missing columns: {', '.join(missing)}")
        for row in reader:
            rows.append(
                {
                    "label": str(row["label"]),
                    "rule_order": float(row["rule_order"]),
                    "complexity_mean": float(row["complexity_mean"]),
                    "complexity_se": float(row["complexity_se"]),
                }
            )

    if not rows:
        raise ValueError(f"no rows for {path}")
    return rows

def _plot(summary_rows: list[dict[str, float | str]]) -> None:
    order = np.asarray([float(row["rule_order"]) for row in summary_rows], dtype=np.float64)
    mean = np.asarray([float(row["complexity_mean"]) for row in summary_rows], dtype=np.float64)
    se = np.asarray([float(row["complexity_se"]) for row in summary_rows], dtype=np.float64)
    labels = [str(row["label"]) for row in summary_rows]
    r, p_value = pearsonr(order, mean)

    FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6.8, 4.4), constrained_layout=True)
    ax.errorbar(order, mean, yerr=se, fmt="o-", color="#284f8f", ecolor="#8aa7d6", capsize=3)
    ax.set_xticks(order)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_xlabel("manual rule")
    ax.set_ylabel("3-NN label-disagreement complexity")
    ax.set_title(f"Manual rule vs complexity (Pearson r={r:.3f}, p={p_value:.1e})")
    ax.grid(True, alpha=0.25)
    fig.savefig(FIGURE_PATH, dpi=240)
    plt.close(fig)

summary_rows = _read_summary(SUMMARY_PATH)
_plot(summary_rows)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 39. dnn_mnist/03_reference_search/label_noise_sweep/reference_quality_by_eta.png

Source image: `03_dnn_mnist/figures/03_reference_search/label_noise_sweep/reference_quality_by_eta.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_eta.csv`
- `03_dnn_mnist/label_noise_sweep/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_ref.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/03_reference_search/label_noise_sweep/reference_quality_by_eta.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_eta.csv', ROOT / '03_dnn_mnist/label_noise_sweep/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_ref.csv']

FIGURE_SUMMARY_PATH = input_paths[0]
FIGURE_PER_REF_PATH = input_paths[1]
FIGURE_ROOT = output_png.parent
FIGURE_PATH = output_png

def _read_inputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not FIGURE_SUMMARY_PATH.exists() or not FIGURE_PER_REF_PATH.exists():
        raise FileNotFoundError("run src/make_summarized_outputs.py first")
    return pd.read_csv(FIGURE_SUMMARY_PATH), pd.read_csv(FIGURE_PER_REF_PATH)

def build_figures() -> None:
    summary, per_ref = _read_inputs()
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

    eta = pd.to_numeric(summary["eta"], errors="coerce").to_numpy(dtype=float)
    test_mean = pd.to_numeric(summary["test_error_mean"], errors="coerce").to_numpy(dtype=float)
    test_sem = pd.to_numeric(summary["test_error_sem"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    norm_mean = pd.to_numeric(summary["theta_norm_mean"], errors="coerce").to_numpy(dtype=float)
    norm_sem = pd.to_numeric(summary["theta_norm_sem"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.4), constrained_layout=True)

    axes[0].errorbar(eta, test_mean, yerr=test_sem, fmt="o-", color="#284f8f", ecolor="#8aa7d6", capsize=3)
    for eta_value, sub in per_ref.groupby("eta", sort=True):
        x = np.full(len(sub), float(eta_value), dtype=float)
        y = pd.to_numeric(sub["test_error"], errors="coerce").to_numpy(dtype=float)
        axes[0].scatter(x, y, s=14, color="#284f8f", alpha=0.20, linewidth=0)
    axes[0].set_xlabel("label noise eta")
    axes[0].set_ylabel("test error")
    axes[0].set_title("Reference generalization")
    axes[0].grid(True, alpha=0.25)

    axes[1].errorbar(eta, norm_mean, yerr=norm_sem, fmt="o-", color="#7a4f1d", ecolor="#c49b6b", capsize=3)
    for eta_value, sub in per_ref.groupby("eta", sort=True):
        x = np.full(len(sub), float(eta_value), dtype=float)
        y = pd.to_numeric(sub["theta_norm"], errors="coerce").to_numpy(dtype=float)
        axes[1].scatter(x, y, s=14, color="#7a4f1d", alpha=0.20, linewidth=0)
    axes[1].set_xlabel("label noise eta")
    axes[1].set_ylabel("theta norm")
    axes[1].set_title("Reference norm")
    axes[1].grid(True, alpha=0.25)

    fig.savefig(FIGURE_PATH, dpi=240)
    plt.close(fig)

build_figures()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 40. dnn_mnist/03_reference_search/manual_rules/reference_quality_by_rule.png

Source image: `03_dnn_mnist/figures/03_reference_search/manual_rules/reference_quality_by_rule.png`

Inputs:
- `03_dnn_mnist/manual_rules/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_rule.csv`
- `03_dnn_mnist/manual_rules/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_ref.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/03_reference_search/manual_rules/reference_quality_by_rule.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_rule.csv', ROOT / '03_dnn_mnist/manual_rules/03_reference_search/summarized_outputs/figure_inputs/reference_quality/reference_quality_by_ref.csv']

SUMMARY_PATH = input_paths[0]
PER_REF_PATH = input_paths[1]
FIGURE_ROOT = output_png.parent
FIGURE_PATH = output_png

def _rule_label(rule: str) -> str:
    return str(rule).replace("_", " ")

def _read_inputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not SUMMARY_PATH.exists() or not PER_REF_PATH.exists():
        raise FileNotFoundError("run src/make_summarized_outputs.py first")
    return pd.read_csv(SUMMARY_PATH), pd.read_csv(PER_REF_PATH)

def build_figures() -> None:
    summary, per_ref = _read_inputs()
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

    labels = [_rule_label(value) for value in summary["rule"].astype(str)]
    x = np.arange(len(summary), dtype=float)
    test_mean = pd.to_numeric(summary["test_error_mean"], errors="coerce").to_numpy(dtype=float)
    test_sem = pd.to_numeric(summary["test_error_sem"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    norm_mean = pd.to_numeric(summary["theta_norm_mean"], errors="coerce").to_numpy(dtype=float)
    norm_sem = pd.to_numeric(summary["theta_norm_sem"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.8), constrained_layout=True)

    axes[0].errorbar(x, test_mean, yerr=test_sem, fmt="o", color="#284f8f", ecolor="#8aa7d6", capsize=3)
    for idx, rule in enumerate(summary["rule"].astype(str)):
        sub = per_ref[per_ref["rule"].astype(str).eq(rule)]
        y = pd.to_numeric(sub["test_error"], errors="coerce").to_numpy(dtype=float)
        jitter = np.linspace(-0.13, 0.13, len(y), dtype=float) if len(y) else np.asarray([], dtype=float)
        axes[0].scatter(np.full(len(y), idx, dtype=float) + jitter, y, s=14, color="#284f8f", alpha=0.22, linewidth=0)
    axes[0].set_ylabel("test error")
    axes[0].set_title("Reference generalization")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=25, ha="right")
    axes[0].grid(True, axis="y", alpha=0.25)

    axes[1].errorbar(x, norm_mean, yerr=norm_sem, fmt="o", color="#3a7f59", ecolor="#9bc3ac", capsize=3)
    for idx, rule in enumerate(summary["rule"].astype(str)):
        sub = per_ref[per_ref["rule"].astype(str).eq(rule)]
        y = pd.to_numeric(sub["theta_norm"], errors="coerce").to_numpy(dtype=float)
        jitter = np.linspace(-0.13, 0.13, len(y), dtype=float) if len(y) else np.asarray([], dtype=float)
        axes[1].scatter(np.full(len(y), idx, dtype=float) + jitter, y, s=14, color="#3a7f59", alpha=0.22, linewidth=0)
    axes[1].set_ylabel("theta L2 norm")
    axes[1].set_title("Reference norm spread")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=25, ha="right")
    axes[1].grid(True, axis="y", alpha=0.25)

    fig.savefig(FIGURE_PATH, dpi=180)
    plt.close(fig)

build_figures()

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 41. dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p05.png

Source image: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p05.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/04_sampling/summarized_outputs/figure_inputs/logZ_split/eta_0p05.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p05.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/04_sampling/summarized_outputs/figure_inputs/logZ_split/eta_0p05.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    eta = float(frame["eta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(eta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, eta={eta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 42. dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p15.png

Source image: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p15.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/04_sampling/summarized_outputs/figure_inputs/logZ_split/eta_0p15.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p15.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/04_sampling/summarized_outputs/figure_inputs/logZ_split/eta_0p15.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    eta = float(frame["eta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(eta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, eta={eta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 43. dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p25.png

Source image: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p25.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/04_sampling/summarized_outputs/figure_inputs/logZ_split/eta_0p25.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/eta_0p25.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/04_sampling/summarized_outputs/figure_inputs/logZ_split/eta_0p25.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    eta = float(frame["eta"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(round(eta * 1000)))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, eta={eta:.2f}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 44. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_001.png

Source image: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/rule_001.png`

Inputs:
- `03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_001.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_001.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_001.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    rule_id = str(frame["rule_id"].iloc[0])
    rule = str(frame["rule"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(rule_id.removeprefix("rule_")))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, {rule_id}: {rule}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 45. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_002.png

Source image: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/rule_002.png`

Inputs:
- `03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_002.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_002.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_002.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    rule_id = str(frame["rule_id"].iloc[0])
    rule = str(frame["rule"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(rule_id.removeprefix("rule_")))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, {rule_id}: {rule}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 46. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_003.png

Source image: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/rule_003.png`

Inputs:
- `03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_003.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_003.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_003.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    rule_id = str(frame["rule_id"].iloc[0])
    rule = str(frame["rule"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(rule_id.removeprefix("rule_")))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, {rule_id}: {rule}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 47. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_004.png

Source image: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/rule_004.png`

Inputs:
- `03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_004.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/rule_004.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/04_sampling/summarized_outputs/figure_inputs/logZ_split/rule_004.csv']

def _radius_label(value: float) -> str:
    return f"{value:g}"

def _xtick_labels(radii: list[float]) -> list[str]:
    if len(radii) <= 24:
        return [_radius_label(value) for value in radii]
    step = max(1, int(np.ceil(len(radii) / 14)))
    return [_radius_label(value) if idx % step == 0 or idx == len(radii) - 1 else "" for idx, value in enumerate(radii)]

def plot_logz_split_distribution(
    input_csv: Path,
    output_png: Path,
    *,
    max_scatter_per_radius: int,
) -> None:
    frame = pd.read_csv(input_csv)
    frame["r"] = pd.to_numeric(frame["r"], errors="coerce")
    frame["signed_split_logZ_per_P_diff"] = pd.to_numeric(frame["signed_split_logZ_per_P_diff"], errors="coerce")
    frame = frame.dropna(subset=["r", "signed_split_logZ_per_P_diff"]).sort_values("r")
    if frame.empty:
        raise ValueError(f"no finite logZ split rows in {input_csv}")

    rule_id = str(frame["rule_id"].iloc[0])
    rule = str(frame["rule"].iloc[0])
    radii = [float(value) for value in sorted(frame["r"].unique())]
    values = [frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float) for radius in radii]
    positions = np.arange(len(radii), dtype=float)
    width = max(10.5, min(18.0, 0.34 * len(radii)))
    fig, ax = plt.subplots(figsize=(width, 5.8))

    violin_positions = positions + 0.16
    parts = ax.violinplot(
        values,
        positions=violin_positions,
        widths=0.34,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for center, body in zip(violin_positions, parts["bodies"]):
        vertices = body.get_paths()[0].vertices
        vertices[:, 0] = np.maximum(vertices[:, 0], center)
        body.set_facecolor("#6f6f6f")
        body.set_edgecolor("none")
        body.set_alpha(0.82)

    rng = np.random.default_rng(1729 + int(rule_id.removeprefix("rule_")))
    color_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.98)), 1.0e-12)
    for idx, radius in enumerate(radii):
        group = frame.loc[np.isclose(frame["r"], radius), "signed_split_logZ_per_P_diff"].to_numpy(float)
        if len(group) > max_scatter_per_radius:
            group = rng.choice(group, size=max_scatter_per_radius, replace=False)
        x = rng.normal(loc=positions[idx] - 0.12, scale=0.045, size=len(group))
        x = np.clip(x, positions[idx] - 0.27, positions[idx] + 0.04)
        ax.scatter(
            x,
            group,
            c=group,
            cmap="coolwarm",
            vmin=-color_extent,
            vmax=color_extent,
            s=18,
            alpha=0.88,
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )

    y_extent = max(float(np.nanquantile(np.abs(frame["signed_split_logZ_per_P_diff"]), 0.995)), 1.0e-12) * 1.12
    ax.set_ylim(-y_extent, y_extent)
    ax.set_xlim(-0.55, len(radii) - 0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(_xtick_labels(radii), rotation=90, fontsize=7)
    ax.set_xlabel("d")
    ax.set_ylabel("signed split logZ diff per P")
    ax.set_title(f"logZ split distributions, {rule_id}: {rule}")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    output_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_png, dpi=220)
    plt.close(fig)

plot_logz_split_distribution(input_paths[0], output_png, max_scatter_per_radius=120)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 48. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/derivative_phi_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COMPLEXITY_SUMMARY_PATH = ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _float(value: object) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if np.isfinite(out) else float("nan")

def _group_curves(
    rows: Iterable[dict[str, str]],
    value_key: str,
    sem_key: str,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    grouped: dict[float, list[tuple[float, float, float]]] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        radius = _float(row.get("radius"))
        value = _float(row.get(value_key))
        sem = _float(row.get(sem_key))
        if np.isfinite(eta) and np.isfinite(radius) and np.isfinite(value):
            grouped.setdefault(eta, []).append((radius, value, sem))

    out: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for eta, values in sorted(grouped.items()):
        ordered = sorted(values)
        radius = np.asarray([x for x, _y, _sem in ordered], dtype=np.float64)
        value = np.asarray([y for _x, y, _sem in ordered], dtype=np.float64)
        sem_values = np.asarray([sem for _x, _y, sem in ordered], dtype=np.float64)
        sem_out = sem_values if np.isfinite(sem_values).any() else None
        out[eta] = (radius, value, sem_out)
    return out

def _plot_curve(
    rows: list[dict[str, str]],
    value_key: str,
    sem_key: str,
    ylabel: str,
    title: str,
    path: Path,
) -> None:
    curves = _group_curves(rows, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for eta, (radius, value, sem) in curves.items():
        color = cmap(norm(eta))
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"$\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'd_delta_phi_energy_dd'
sem_key = 'd_delta_phi_energy_dd_sem'
ylabel = '$d\\phi/dd$'
title = 'MNIST label-noise derivative of $\\phi(d)$'
rows = _read_csv(input_paths[0])
_plot_curve(rows, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 49. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_energetic_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/derivative_phi_energetic_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_energetic_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COMPLEXITY_SUMMARY_PATH = ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _float(value: object) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if np.isfinite(out) else float("nan")

def _group_curves(
    rows: Iterable[dict[str, str]],
    value_key: str,
    sem_key: str,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    grouped: dict[float, list[tuple[float, float, float]]] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        radius = _float(row.get("radius"))
        value = _float(row.get(value_key))
        sem = _float(row.get(sem_key))
        if np.isfinite(eta) and np.isfinite(radius) and np.isfinite(value):
            grouped.setdefault(eta, []).append((radius, value, sem))

    out: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for eta, values in sorted(grouped.items()):
        ordered = sorted(values)
        radius = np.asarray([x for x, _y, _sem in ordered], dtype=np.float64)
        value = np.asarray([y for _x, y, _sem in ordered], dtype=np.float64)
        sem_values = np.asarray([sem for _x, _y, sem in ordered], dtype=np.float64)
        sem_out = sem_values if np.isfinite(sem_values).any() else None
        out[eta] = (radius, value, sem_out)
    return out

def _plot_curve(
    rows: list[dict[str, str]],
    value_key: str,
    sem_key: str,
    ylabel: str,
    title: str,
    path: Path,
) -> None:
    curves = _group_curves(rows, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for eta, (radius, value, sem) in curves.items():
        color = cmap(norm(eta))
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"$\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'd_phi_energy_direct_dd'
sem_key = 'd_phi_energy_direct_dd_sem'
ylabel = 'energetic $d\\phi/dd$'
title = 'MNIST label-noise direct energetic derivative'
rows = _read_csv(input_paths[0])
_plot_curve(rows, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 50. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_complexity.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_complexity.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv`
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_complexity.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COMPLEXITY_SUMMARY_PATH = ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _float(value: object) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if np.isfinite(out) else float("nan")

def _eta_complexity_lookup() -> dict[float, float]:
    rows = _read_csv(COMPLEXITY_SUMMARY_PATH)
    lookup: dict[float, float] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        complexity = _float(row.get("complexity_mean"))
        if np.isfinite(eta) and np.isfinite(complexity):
            lookup[round(float(eta), 10)] = float(complexity)
    return lookup

def _group_curves(
    rows: Iterable[dict[str, str]],
    value_key: str,
    sem_key: str,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    grouped: dict[float, list[tuple[float, float, float]]] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        radius = _float(row.get("radius"))
        value = _float(row.get(value_key))
        sem = _float(row.get(sem_key))
        if np.isfinite(eta) and np.isfinite(radius) and np.isfinite(value):
            grouped.setdefault(eta, []).append((radius, value, sem))

    out: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for eta, values in sorted(grouped.items()):
        ordered = sorted(values)
        radius = np.asarray([x for x, _y, _sem in ordered], dtype=np.float64)
        value = np.asarray([y for _x, y, _sem in ordered], dtype=np.float64)
        sem_values = np.asarray([sem for _x, _y, sem in ordered], dtype=np.float64)
        sem_out = sem_values if np.isfinite(sem_values).any() else None
        out[eta] = (radius, value, sem_out)
    return out

def _plot_phase(name: str, x_key: str, x_label: str, title: str, output_name: str) -> None:
    root = FIGURE_INPUT_ROOT / name
    phase = _read_csv(root / f"{name}.csv")
    curves = _read_csv(root / "phase_derivative_curves.csv")
    curve_groups = _group_curves(curves, "dphi_dr_smooth_mean", "dphi_dr_smooth_sem")
    if not curve_groups:
        raise ValueError(f"no finite derivative rows to plot for {name}")

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curve_groups), max(curve_groups))
    has_band = False
    for eta, (radius, value, sem) in curve_groups.items():
        color = cmap(norm(eta))
        ax_left.plot(radius, value, linewidth=1.1, alpha=0.85, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax_left.fill_between(radius, value - err, value + err, color=color, alpha=0.09, linewidth=0)
            has_band = True

    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel(r"energetic $d\phi/dd$")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.22)
    if has_band:
        ax_left.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax_left.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    if x_key == "nmstv":
        complexity = _eta_complexity_lookup()
        x = np.asarray(
            [complexity.get(round(_float(row.get("eta")), 10), float("nan")) for row in phase],
            dtype=np.float64,
        )
    else:
        x = np.asarray([_float(row.get(x_key)) for row in phase], dtype=np.float64)
    y = np.asarray([_float(row.get("A_kappa_mean")) for row in phase], dtype=np.float64)
    yerr = np.asarray([_float(row.get("A_kappa_sem")) for row in phase], dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    yerr = yerr[mask]
    order = np.argsort(x)
    ax_right.errorbar(
        x[order],
        y[order],
        yerr=np.nan_to_num(yerr[order], nan=0.0),
        marker="o",
        linewidth=1.5,
        capsize=2.5,
        color="#7a3e9d",
    )
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title(title)
    ax_right.grid(True, alpha=0.25)

    out_path = FIGURE_ROOT / output_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=240)
    plt.close(fig)

_plot_phase('phase_like_A_by_complexity', 'nmstv', '3-NN MNIST complexity', 'A measure by complexity', output_png.name)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 51. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_eta.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_eta.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_like_A_by_eta.csv`
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_derivative_curves.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_eta.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_like_A_by_eta.csv', ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_derivative_curves.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COMPLEXITY_SUMMARY_PATH = ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _float(value: object) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if np.isfinite(out) else float("nan")

def _eta_complexity_lookup() -> dict[float, float]:
    rows = _read_csv(COMPLEXITY_SUMMARY_PATH)
    lookup: dict[float, float] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        complexity = _float(row.get("complexity_mean"))
        if np.isfinite(eta) and np.isfinite(complexity):
            lookup[round(float(eta), 10)] = float(complexity)
    return lookup

def _group_curves(
    rows: Iterable[dict[str, str]],
    value_key: str,
    sem_key: str,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    grouped: dict[float, list[tuple[float, float, float]]] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        radius = _float(row.get("radius"))
        value = _float(row.get(value_key))
        sem = _float(row.get(sem_key))
        if np.isfinite(eta) and np.isfinite(radius) and np.isfinite(value):
            grouped.setdefault(eta, []).append((radius, value, sem))

    out: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for eta, values in sorted(grouped.items()):
        ordered = sorted(values)
        radius = np.asarray([x for x, _y, _sem in ordered], dtype=np.float64)
        value = np.asarray([y for _x, y, _sem in ordered], dtype=np.float64)
        sem_values = np.asarray([sem for _x, _y, sem in ordered], dtype=np.float64)
        sem_out = sem_values if np.isfinite(sem_values).any() else None
        out[eta] = (radius, value, sem_out)
    return out

def _plot_phase(name: str, x_key: str, x_label: str, title: str, output_name: str) -> None:
    root = FIGURE_INPUT_ROOT / name
    phase = _read_csv(root / f"{name}.csv")
    curves = _read_csv(root / "phase_derivative_curves.csv")
    curve_groups = _group_curves(curves, "dphi_dr_smooth_mean", "dphi_dr_smooth_sem")
    if not curve_groups:
        raise ValueError(f"no finite derivative rows to plot for {name}")

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curve_groups), max(curve_groups))
    has_band = False
    for eta, (radius, value, sem) in curve_groups.items():
        color = cmap(norm(eta))
        ax_left.plot(radius, value, linewidth=1.1, alpha=0.85, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax_left.fill_between(radius, value - err, value + err, color=color, alpha=0.09, linewidth=0)
            has_band = True

    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel(r"energetic $d\phi/dd$")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.22)
    if has_band:
        ax_left.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax_left.transAxes,
            fontsize=7.5,
            color="0.35",
        )

    if x_key == "nmstv":
        complexity = _eta_complexity_lookup()
        x = np.asarray(
            [complexity.get(round(_float(row.get("eta")), 10), float("nan")) for row in phase],
            dtype=np.float64,
        )
    else:
        x = np.asarray([_float(row.get(x_key)) for row in phase], dtype=np.float64)
    y = np.asarray([_float(row.get("A_kappa_mean")) for row in phase], dtype=np.float64)
    yerr = np.asarray([_float(row.get("A_kappa_sem")) for row in phase], dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    yerr = yerr[mask]
    order = np.argsort(x)
    ax_right.errorbar(
        x[order],
        y[order],
        yerr=np.nan_to_num(yerr[order], nan=0.0),
        marker="o",
        linewidth=1.5,
        capsize=2.5,
        color="#7a3e9d",
    )
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title(title)
    ax_right.grid(True, alpha=0.25)

    out_path = FIGURE_ROOT / output_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=240)
    plt.close(fig)

_plot_phase('phase_like_A_by_eta', 'eta', '$\\eta$', 'A measure by eta', output_png.name)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 52. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phi_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COMPLEXITY_SUMMARY_PATH = ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _float(value: object) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if np.isfinite(out) else float("nan")

def _group_curves(
    rows: Iterable[dict[str, str]],
    value_key: str,
    sem_key: str,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    grouped: dict[float, list[tuple[float, float, float]]] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        radius = _float(row.get("radius"))
        value = _float(row.get(value_key))
        sem = _float(row.get(sem_key))
        if np.isfinite(eta) and np.isfinite(radius) and np.isfinite(value):
            grouped.setdefault(eta, []).append((radius, value, sem))

    out: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for eta, values in sorted(grouped.items()):
        ordered = sorted(values)
        radius = np.asarray([x for x, _y, _sem in ordered], dtype=np.float64)
        value = np.asarray([y for _x, y, _sem in ordered], dtype=np.float64)
        sem_values = np.asarray([sem for _x, _y, sem in ordered], dtype=np.float64)
        sem_out = sem_values if np.isfinite(sem_values).any() else None
        out[eta] = (radius, value, sem_out)
    return out

def _plot_curve(
    rows: list[dict[str, str]],
    value_key: str,
    sem_key: str,
    ylabel: str,
    title: str,
    path: Path,
) -> None:
    curves = _group_curves(rows, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for eta, (radius, value, sem) in curves.items():
        color = cmap(norm(eta))
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"$\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'delta_phi_energy_mean'
sem_key = 'delta_phi_energy_sem'
ylabel = '$\\phi(d)-\\phi(d_0)$'
title = 'MNIST label-noise $\\phi(d)$'
rows = _read_csv(input_paths[0])
_plot_curve(rows, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 53. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_energetic_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phi_energetic_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_energetic_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COMPLEXITY_SUMMARY_PATH = ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'

def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _float(value: object) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if np.isfinite(out) else float("nan")

def _group_curves(
    rows: Iterable[dict[str, str]],
    value_key: str,
    sem_key: str,
) -> dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]]:
    grouped: dict[float, list[tuple[float, float, float]]] = {}
    for row in rows:
        eta = _float(row.get("eta"))
        radius = _float(row.get("radius"))
        value = _float(row.get(value_key))
        sem = _float(row.get(sem_key))
        if np.isfinite(eta) and np.isfinite(radius) and np.isfinite(value):
            grouped.setdefault(eta, []).append((radius, value, sem))

    out: dict[float, tuple[np.ndarray, np.ndarray, np.ndarray | None]] = {}
    for eta, values in sorted(grouped.items()):
        ordered = sorted(values)
        radius = np.asarray([x for x, _y, _sem in ordered], dtype=np.float64)
        value = np.asarray([y for _x, y, _sem in ordered], dtype=np.float64)
        sem_values = np.asarray([sem for _x, _y, sem in ordered], dtype=np.float64)
        sem_out = sem_values if np.isfinite(sem_values).any() else None
        out[eta] = (radius, value, sem_out)
    return out

def _plot_curve(
    rows: list[dict[str, str]],
    value_key: str,
    sem_key: str,
    ylabel: str,
    title: str,
    path: Path,
) -> None:
    curves = _group_curves(rows, value_key, sem_key)
    if not curves:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(curves), max(curves))
    has_band = False

    for eta, (radius, value, sem) in curves.items():
        color = cmap(norm(eta))
        ax.plot(radius, value, linewidth=1.35, alpha=0.9, color=color)
        if sem is not None:
            err = np.nan_to_num(sem, nan=0.0)
            ax.fill_between(radius, value - err, value + err, color=color, alpha=0.10, linewidth=0)
            has_band = True

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"$\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.22)
    if has_band:
        ax.text(
            0.01,
            0.02,
            "band: mean +/- standard error",
            transform=ax.transAxes,
            fontsize=7.5,
            color="0.35",
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'phi_energy_raw_mean'
sem_key = 'phi_energy_raw_sem'
ylabel = 'energetic $\\phi(d)$'
title = 'MNIST label-noise energetic $\\phi(d)$'
rows = _read_csv(input_paths[0])
_plot_curve(rows, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 54. dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/derivative_phi_d_curve.png`

Inputs:
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COLORS = {
    'rule_001': '#0072B2',
    'rule_002': '#009E73',
    'rule_003': '#D55E00',
    'rule_004': '#CC79A7',
}

def _plot_curves(frame: pd.DataFrame, value: str, sem: str, ylabel: str, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    for rule_id, sub in frame.groupby("rule_id", sort=True):
        sub = sub.sort_values("radius")
        label = str(sub["rule_label"].iloc[0])
        ax.plot(sub["radius"], sub[value], linewidth=1.8, color=COLORS.get(rule_id), label=label)
        if sem in sub.columns:
            lower = sub[value] - sub[sem]
            upper = sub[value] + sub[sem]
            ax.fill_between(sub["radius"], lower, upper, color=COLORS.get(rule_id), alpha=0.14, linewidth=0)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(frameon=False, fontsize=8)
    ax.text(
        0.01,
        0.02,
        "band: mean +/- SE across references",
        transform=ax.transAxes,
        fontsize=7.5,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'd_delta_phi_energy_direct_dd_unit_mean'
sem_key = 'd_delta_phi_energy_direct_dd_unit_sem'
ylabel = 'd phi / dd'
title = 'MNIST manual-rule direct derivative of phi(d)'
frame = pd.read_csv(input_paths[0])
_plot_curves(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 55. dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_energetic_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/derivative_phi_energetic_d_curve.png`

Inputs:
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_energetic_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COLORS = {
    'rule_001': '#0072B2',
    'rule_002': '#009E73',
    'rule_003': '#D55E00',
    'rule_004': '#CC79A7',
}

def _plot_curves(frame: pd.DataFrame, value: str, sem: str, ylabel: str, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    for rule_id, sub in frame.groupby("rule_id", sort=True):
        sub = sub.sort_values("radius")
        label = str(sub["rule_label"].iloc[0])
        ax.plot(sub["radius"], sub[value], linewidth=1.8, color=COLORS.get(rule_id), label=label)
        if sem in sub.columns:
            lower = sub[value] - sub[sem]
            upper = sub[value] + sub[sem]
            ax.fill_between(sub["radius"], lower, upper, color=COLORS.get(rule_id), alpha=0.14, linewidth=0)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(frameon=False, fontsize=8)
    ax.text(
        0.01,
        0.02,
        "band: mean +/- SE across references",
        transform=ax.transAxes,
        fontsize=7.5,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'd_phi_energy_direct_dd_unit_mean'
sem_key = 'd_phi_energy_direct_dd_unit_sem'
ylabel = 'energetic d phi / dd'
title = 'MNIST manual-rule direct energetic derivative'
frame = pd.read_csv(input_paths[0])
_plot_curves(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 56. dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_complexity.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phase_like_A_by_complexity.png`

Inputs:
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_complexity.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COLORS = {
    'rule_001': '#0072B2',
    'rule_002': '#009E73',
    'rule_003': '#D55E00',
    'rule_004': '#CC79A7',
}

def _plot_phase(x_key: str, x_label: str, output_name: str) -> None:
    phase = pd.read_csv(FIGURE_INPUT_ROOT / output_name / f"{output_name}.csv")
    curves = pd.read_csv(FIGURE_INPUT_ROOT / output_name / "phase_derivative_curves.csv")
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
    for rule_id, sub in curves.groupby("rule_id", sort=True):
        sub = sub.sort_values("radius")
        label = str(sub["rule_label"].iloc[0])
        ax_left.plot(
            sub["radius"],
            sub["dphi_dr_smooth_mean"],
            linewidth=1.5,
            color=COLORS.get(rule_id),
            label=label,
        )
        if "dphi_dr_smooth_sem" in sub.columns:
            x = sub["radius"].to_numpy(dtype=float)
            y = sub["dphi_dr_smooth_mean"].to_numpy(dtype=float)
            err = sub["dphi_dr_smooth_sem"].fillna(0.0).to_numpy(dtype=float)
            ax_left.fill_between(x, y - err, y + err, color=COLORS.get(rule_id), alpha=0.10, linewidth=0)
    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel("energetic d phi / dd")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.24)
    ax_left.legend(frameon=False, fontsize=8)
    ax_left.text(
        0.01,
        0.02,
        "band: mean +/- SE across references",
        transform=ax_left.transAxes,
        fontsize=7.5,
        color="0.35",
    )

    phase = phase.sort_values(x_key)
    ax_right.errorbar(
        phase[x_key],
        phase["A_kappa_mean"],
        yerr=phase["A_kappa_sem"],
        marker="o",
        linewidth=1.5,
        capsize=2.8,
        color="#4C78A8",
    )
    for _, row in phase.iterrows():
        ax_right.annotate(
            str(row["rule_label"]),
            (row[x_key], row["A_kappa_mean"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8,
        )
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title("A-measure phase-like plot")
    ax_right.grid(True, alpha=0.24)
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_ROOT / f"{output_name}.png", dpi=240)
    plt.close(fig)

_plot_phase('nmstv_mean', '3-NN MNIST complexity', 'phase_like_A_by_complexity')

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 57. dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_rule.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phase_like_A_by_rule.png`

Inputs:
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_like_A_by_rule.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_derivative_curves.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_rule.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_like_A_by_rule.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_derivative_curves.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COLORS = {
    'rule_001': '#0072B2',
    'rule_002': '#009E73',
    'rule_003': '#D55E00',
    'rule_004': '#CC79A7',
}

def _plot_phase(x_key: str, x_label: str, output_name: str) -> None:
    phase = pd.read_csv(FIGURE_INPUT_ROOT / output_name / f"{output_name}.csv")
    curves = pd.read_csv(FIGURE_INPUT_ROOT / output_name / "phase_derivative_curves.csv")
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
    for rule_id, sub in curves.groupby("rule_id", sort=True):
        sub = sub.sort_values("radius")
        label = str(sub["rule_label"].iloc[0])
        ax_left.plot(
            sub["radius"],
            sub["dphi_dr_smooth_mean"],
            linewidth=1.5,
            color=COLORS.get(rule_id),
            label=label,
        )
        if "dphi_dr_smooth_sem" in sub.columns:
            x = sub["radius"].to_numpy(dtype=float)
            y = sub["dphi_dr_smooth_mean"].to_numpy(dtype=float)
            err = sub["dphi_dr_smooth_sem"].fillna(0.0).to_numpy(dtype=float)
            ax_left.fill_between(x, y - err, y + err, color=COLORS.get(rule_id), alpha=0.10, linewidth=0)
    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel("energetic d phi / dd")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.24)
    ax_left.legend(frameon=False, fontsize=8)
    ax_left.text(
        0.01,
        0.02,
        "band: mean +/- SE across references",
        transform=ax_left.transAxes,
        fontsize=7.5,
        color="0.35",
    )

    phase = phase.sort_values(x_key)
    ax_right.errorbar(
        phase[x_key],
        phase["A_kappa_mean"],
        yerr=phase["A_kappa_sem"],
        marker="o",
        linewidth=1.5,
        capsize=2.8,
        color="#4C78A8",
    )
    for _, row in phase.iterrows():
        ax_right.annotate(
            str(row["rule_label"]),
            (row[x_key], row["A_kappa_mean"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8,
        )
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title("A-measure phase-like plot")
    ax_right.grid(True, alpha=0.24)
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_ROOT / f"{output_name}.png", dpi=240)
    plt.close(fig)

_plot_phase('rule_order', 'manual-rule order', 'phase_like_A_by_rule')

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 58. dnn_mnist/05_proxy_local_entropy/manual_rules/phi_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phi_d_curve.png`

Inputs:
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/manual_rules/phi_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COLORS = {
    'rule_001': '#0072B2',
    'rule_002': '#009E73',
    'rule_003': '#D55E00',
    'rule_004': '#CC79A7',
}

def _plot_curves(frame: pd.DataFrame, value: str, sem: str, ylabel: str, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    for rule_id, sub in frame.groupby("rule_id", sort=True):
        sub = sub.sort_values("radius")
        label = str(sub["rule_label"].iloc[0])
        ax.plot(sub["radius"], sub[value], linewidth=1.8, color=COLORS.get(rule_id), label=label)
        if sem in sub.columns:
            lower = sub[value] - sub[sem]
            upper = sub[value] + sub[sem]
            ax.fill_between(sub["radius"], lower, upper, color=COLORS.get(rule_id), alpha=0.14, linewidth=0)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(frameon=False, fontsize=8)
    ax.text(
        0.01,
        0.02,
        "band: mean +/- SE across references",
        transform=ax.transAxes,
        fontsize=7.5,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'delta_phi_energy_unit_mean'
sem_key = 'delta_phi_energy_unit_sem'
ylabel = 'phi(d) - phi(d0)'
title = 'MNIST manual-rule phi(d)'
frame = pd.read_csv(input_paths[0])
_plot_curves(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 59. dnn_mnist/05_proxy_local_entropy/manual_rules/phi_energetic_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phi_energetic_d_curve.png`

Inputs:
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/manual_rules/phi_energetic_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv']

FIGURE_INPUT_ROOT = ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs'
FIGURE_ROOT = output_png.parent
COLORS = {
    'rule_001': '#0072B2',
    'rule_002': '#009E73',
    'rule_003': '#D55E00',
    'rule_004': '#CC79A7',
}

def _plot_curves(frame: pd.DataFrame, value: str, sem: str, ylabel: str, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    for rule_id, sub in frame.groupby("rule_id", sort=True):
        sub = sub.sort_values("radius")
        label = str(sub["rule_label"].iloc[0])
        ax.plot(sub["radius"], sub[value], linewidth=1.8, color=COLORS.get(rule_id), label=label)
        if sem in sub.columns:
            lower = sub[value] - sub[sem]
            upper = sub[value] + sub[sem]
            ax.fill_between(sub["radius"], lower, upper, color=COLORS.get(rule_id), alpha=0.14, linewidth=0)
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(frameon=False, fontsize=8)
    ax.text(
        0.01,
        0.02,
        "band: mean +/- SE across references",
        transform=ax.transAxes,
        fontsize=7.5,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

value_key = 'phi_energy_raw_mean'
sem_key = 'phi_energy_raw_sem'
ylabel = 'energetic phi(d)'
title = 'MNIST manual-rule energetic phi(d)'
frame = pd.read_csv(input_paths[0])
_plot_curves(frame, value_key, sem_key, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 60. dnn_mnist/05_proxy_local_entropy/merged/derivative_phi_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/merged/derivative_phi_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/merged/derivative_phi_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
PLE_INPUT_ROOTS = {
    'label_noise_sweep': DNN_ROOT / 'label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs',
    'manual_rules': DNN_ROOT / 'manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs',
}
ETA_COMPLEXITY_PATH = DNN_ROOT / 'label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'
MANUAL_COMPLEXITY_PATH = DNN_ROOT / 'manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv'
ENDPOINT_ETA = {'real_even_odd': 0.0, 'random_label': 0.5}
ENDPOINT_LABEL = {'real_even_odd': 'even_odd (eta 0.00)', 'random_label': 'random (eta 0.50)'}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def finite_or_zero(values: pd.Series) -> np.ndarray:
    return pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype=float)

def label_noise_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    out = pd.DataFrame(
        {
            "eta": pd.to_numeric(frame["eta"], errors="coerce"),
            "condition": frame["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def manual_endpoint_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    frame = frame[frame["rule_name"].isin(ENDPOINT_ETA)].copy()
    out = pd.DataFrame(
        {
            "eta": frame["rule_name"].map(ENDPOINT_ETA),
            "condition": frame["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def merged_curve_frame(
    name: str,
    label_value_col: str,
    label_sem_col: str,
    manual_value_col: str,
    manual_sem_col: str,
) -> tuple[pd.DataFrame, tuple[Path, ...]]:
    label_frame, label_path = label_noise_curve_frame(name, label_value_col, label_sem_col)
    manual_frame, manual_path = manual_endpoint_curve_frame(name, manual_value_col, manual_sem_col)
    frame = pd.concat([manual_frame, label_frame], ignore_index=True).sort_values(["eta", "radius"])
    return frame, (label_path, manual_path)

def plot_merged_curves(frame: pd.DataFrame, ylabel: str, title: str, path: Path) -> None:
    if frame.empty:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(8.2, 5.1), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = Normalize(0.0, 0.5)
    handles = []

    for eta, group in frame.groupby("eta", sort=True):
        group = group.sort_values("radius")
        color = cmap(norm(float(eta)))
        source = str(group["source"].iloc[0])
        label = str(group["condition"].iloc[0])
        linestyle = "-" if source == "manual_endpoint" else "--"
        linewidth = 2.1 if source == "manual_endpoint" else 1.65
        x = group["radius"].to_numpy(dtype=float)
        y = group["value"].to_numpy(dtype=float)
        line = ax.plot(x, y, color=color, linestyle=linestyle, linewidth=linewidth, label=label)[0]
        handles.append(line)
        if "sem" in group.columns:
            err = finite_or_zero(group["sem"])
            ax.fill_between(x, y - err, y + err, color=color, alpha=0.11, linewidth=0)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"aligned $\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(handles=handles, frameon=False, fontsize=8.0, ncol=2)
    ax.text(
        0.01,
        0.02,
        "solid: manual endpoints; dashed: label-noise sweep; band: mean +/- SE",
        transform=ax.transAxes,
        fontsize=7.2,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

name = 'derivative_phi_d_curve'
label_value_col = 'd_delta_phi_energy_dd'
label_sem_col = 'd_delta_phi_energy_dd_sem'
manual_value_col = 'd_delta_phi_energy_direct_dd_unit_mean'
manual_sem_col = 'd_delta_phi_energy_direct_dd_unit_sem'
ylabel = '$d\\phi/dd$'
title = 'Merged MNIST derivative of phi(d): endpoints + eta sweep'
frame, source_inputs = merged_curve_frame(name, label_value_col, label_sem_col, manual_value_col, manual_sem_col)
plot_merged_curves(frame, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 61. dnn_mnist/05_proxy_local_entropy/merged/derivative_phi_energetic_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/merged/derivative_phi_energetic_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/merged/derivative_phi_energetic_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
PLE_INPUT_ROOTS = {
    'label_noise_sweep': DNN_ROOT / 'label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs',
    'manual_rules': DNN_ROOT / 'manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs',
}
ETA_COMPLEXITY_PATH = DNN_ROOT / 'label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'
MANUAL_COMPLEXITY_PATH = DNN_ROOT / 'manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv'
ENDPOINT_ETA = {'real_even_odd': 0.0, 'random_label': 0.5}
ENDPOINT_LABEL = {'real_even_odd': 'even_odd (eta 0.00)', 'random_label': 'random (eta 0.50)'}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def finite_or_zero(values: pd.Series) -> np.ndarray:
    return pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype=float)

def label_noise_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    out = pd.DataFrame(
        {
            "eta": pd.to_numeric(frame["eta"], errors="coerce"),
            "condition": frame["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def manual_endpoint_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    frame = frame[frame["rule_name"].isin(ENDPOINT_ETA)].copy()
    out = pd.DataFrame(
        {
            "eta": frame["rule_name"].map(ENDPOINT_ETA),
            "condition": frame["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def merged_curve_frame(
    name: str,
    label_value_col: str,
    label_sem_col: str,
    manual_value_col: str,
    manual_sem_col: str,
) -> tuple[pd.DataFrame, tuple[Path, ...]]:
    label_frame, label_path = label_noise_curve_frame(name, label_value_col, label_sem_col)
    manual_frame, manual_path = manual_endpoint_curve_frame(name, manual_value_col, manual_sem_col)
    frame = pd.concat([manual_frame, label_frame], ignore_index=True).sort_values(["eta", "radius"])
    return frame, (label_path, manual_path)

def plot_merged_curves(frame: pd.DataFrame, ylabel: str, title: str, path: Path) -> None:
    if frame.empty:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(8.2, 5.1), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = Normalize(0.0, 0.5)
    handles = []

    for eta, group in frame.groupby("eta", sort=True):
        group = group.sort_values("radius")
        color = cmap(norm(float(eta)))
        source = str(group["source"].iloc[0])
        label = str(group["condition"].iloc[0])
        linestyle = "-" if source == "manual_endpoint" else "--"
        linewidth = 2.1 if source == "manual_endpoint" else 1.65
        x = group["radius"].to_numpy(dtype=float)
        y = group["value"].to_numpy(dtype=float)
        line = ax.plot(x, y, color=color, linestyle=linestyle, linewidth=linewidth, label=label)[0]
        handles.append(line)
        if "sem" in group.columns:
            err = finite_or_zero(group["sem"])
            ax.fill_between(x, y - err, y + err, color=color, alpha=0.11, linewidth=0)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"aligned $\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(handles=handles, frameon=False, fontsize=8.0, ncol=2)
    ax.text(
        0.01,
        0.02,
        "solid: manual endpoints; dashed: label-noise sweep; band: mean +/- SE",
        transform=ax.transAxes,
        fontsize=7.2,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

name = 'derivative_phi_energetic_d_curve'
label_value_col = 'd_phi_energy_direct_dd'
label_sem_col = 'd_phi_energy_direct_dd_sem'
manual_value_col = 'd_phi_energy_direct_dd_unit_mean'
manual_sem_col = 'd_phi_energy_direct_dd_unit_sem'
ylabel = 'energetic $d\\phi/dd$'
title = 'Merged MNIST energetic derivative: endpoints + eta sweep'
frame, source_inputs = merged_curve_frame(name, label_value_col, label_sem_col, manual_value_col, manual_sem_col)
plot_merged_curves(frame, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 62. dnn_mnist/05_proxy_local_entropy/merged/phase_like_A_by_complexity.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/merged/phase_like_A_by_complexity.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv`
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv`
- `03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv`
- `03_dnn_mnist/manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/merged/phase_like_A_by_complexity.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv', ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv', ROOT / '03_dnn_mnist/manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
PLE_INPUT_ROOTS = {
    'label_noise_sweep': DNN_ROOT / 'label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs',
    'manual_rules': DNN_ROOT / 'manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs',
}
ETA_COMPLEXITY_PATH = DNN_ROOT / 'label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'
MANUAL_COMPLEXITY_PATH = DNN_ROOT / 'manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv'
ENDPOINT_ETA = {'real_even_odd': 0.0, 'random_label': 0.5}
ENDPOINT_LABEL = {'real_even_odd': 'even_odd (eta 0.00)', 'random_label': 'random (eta 0.50)'}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def finite_or_zero(values: pd.Series) -> np.ndarray:
    return pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype=float)

def eta_complexity_map() -> dict[float, float]:
    frame = pd.read_csv(require_file(ETA_COMPLEXITY_PATH))
    return {
        round(float(row["eta"]), 10): float(row["complexity_mean"])
        for _, row in frame.dropna(subset=["eta", "complexity_mean"]).iterrows()
    }

def manual_complexity_map() -> dict[str, float]:
    frame = pd.read_csv(require_file(MANUAL_COMPLEXITY_PATH))
    return {
        str(row["rule_name"]): float(row["complexity_mean"])
        for _, row in frame.dropna(subset=["rule_name", "complexity_mean"]).iterrows()
    }

def map_eta_complexity(values: pd.Series) -> pd.Series:
    lookup = eta_complexity_map()
    eta = pd.to_numeric(values, errors="coerce")
    missing = sorted({float(value) for value in eta.dropna() if round(float(value), 10) not in lookup})
    if missing:
        raise ValueError(f"{ETA_COMPLEXITY_PATH} is missing eta values required by PLE phase inputs: {missing}")
    return eta.map(lambda value: lookup[round(float(value), 10)] if pd.notna(value) else np.nan)

def map_manual_complexity(values: pd.Series) -> pd.Series:
    lookup = manual_complexity_map()
    rules = values.astype(str)
    missing = sorted(set(rules.dropna()).difference(lookup))
    if missing:
        raise ValueError(f"{MANUAL_COMPLEXITY_PATH} is missing rules required by PLE phase inputs: {missing}")
    return rules.map(lookup)

def label_phase_frames(name: str) -> tuple[pd.DataFrame, pd.DataFrame, tuple[Path, ...]]:
    phase_path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / f"{name}.csv")
    curves_path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / "phase_derivative_curves.csv")
    phase = pd.read_csv(phase_path)
    curves = pd.read_csv(curves_path)

    phase_out = pd.DataFrame(
        {
            "eta": pd.to_numeric(phase["eta"], errors="coerce"),
            "complexity": map_eta_complexity(phase["eta"]),
            "condition": phase["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "A_kappa_mean": pd.to_numeric(phase["A_kappa_mean"], errors="coerce"),
            "A_kappa_sem": pd.to_numeric(phase["A_kappa_sem"], errors="coerce"),
        }
    )
    curves_out = pd.DataFrame(
        {
            "eta": pd.to_numeric(curves["eta"], errors="coerce"),
            "condition": curves["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "radius": pd.to_numeric(curves["radius"], errors="coerce"),
            "value": pd.to_numeric(curves["dphi_dr_smooth_mean"], errors="coerce"),
            "sem": pd.to_numeric(curves["dphi_dr_smooth_sem"], errors="coerce"),
        }
    )
    return (
        phase_out.dropna(subset=["eta", "A_kappa_mean"]),
        curves_out.dropna(subset=["eta", "radius", "value"]),
        (phase_path, curves_path, ETA_COMPLEXITY_PATH),
    )

def manual_endpoint_phase_frames(name: str) -> tuple[pd.DataFrame, pd.DataFrame, tuple[Path, ...]]:
    phase_path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / f"{name}.csv")
    curves_path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / "phase_derivative_curves.csv")
    phase = pd.read_csv(phase_path)
    curves = pd.read_csv(curves_path)
    phase = phase[phase["rule_name"].isin(ENDPOINT_ETA)].copy()
    curves = curves[curves["rule_name"].isin(ENDPOINT_ETA)].copy()

    phase_out = pd.DataFrame(
        {
            "eta": phase["rule_name"].map(ENDPOINT_ETA),
            "complexity": map_manual_complexity(phase["rule_name"]),
            "condition": phase["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "A_kappa_mean": pd.to_numeric(phase["A_kappa_mean"], errors="coerce"),
            "A_kappa_sem": pd.to_numeric(phase["A_kappa_sem"], errors="coerce"),
        }
    )
    curves_out = pd.DataFrame(
        {
            "eta": curves["rule_name"].map(ENDPOINT_ETA),
            "condition": curves["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "radius": pd.to_numeric(curves["radius"], errors="coerce"),
            "value": pd.to_numeric(curves["dphi_dr_smooth_mean"], errors="coerce"),
            "sem": pd.to_numeric(curves["dphi_dr_smooth_sem"], errors="coerce"),
        }
    )
    return (
        phase_out.dropna(subset=["eta", "A_kappa_mean"]),
        curves_out.dropna(subset=["eta", "radius", "value"]),
        (phase_path, curves_path, MANUAL_COMPLEXITY_PATH),
    )

def plot_merged_phase(
    phase: pd.DataFrame,
    curves: pd.DataFrame,
    path: Path,
    *,
    x_col: str,
    x_label: str,
    right_title: str,
    eta_xlim: bool = False,
) -> None:
    if phase.empty or curves.empty:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.4, 4.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = Normalize(0.0, 0.5)
    handles = []

    for eta, group in curves.groupby("eta", sort=True):
        group = group.sort_values("radius")
        color = cmap(norm(float(eta)))
        source = str(group["source"].iloc[0])
        label = str(group["condition"].iloc[0])
        linestyle = "-" if source == "manual_endpoint" else "--"
        linewidth = 2.1 if source == "manual_endpoint" else 1.55
        x = group["radius"].to_numpy(dtype=float)
        y = group["value"].to_numpy(dtype=float)
        line = ax_left.plot(x, y, color=color, linestyle=linestyle, linewidth=linewidth, label=label)[0]
        handles.append(line)
        err = finite_or_zero(group["sem"])
        ax_left.fill_between(x, y - err, y + err, color=color, alpha=0.10, linewidth=0)

    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel(r"energetic $d\phi/dd$")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.24)
    ax_left.legend(handles=handles, frameon=False, fontsize=8.0, ncol=1)
    ax_left.text(
        0.01,
        0.02,
        "solid endpoints; dashed eta sweep",
        transform=ax_left.transAxes,
        fontsize=7.2,
        color="0.35",
    )

    phase = phase.sort_values(x_col)
    colors = [cmap(norm(float(eta))) for eta in phase["eta"]]
    ax_right.errorbar(
        phase[x_col],
        phase["A_kappa_mean"],
        yerr=finite_or_zero(phase["A_kappa_sem"]),
        fmt="none",
        ecolor="0.35",
        elinewidth=1.0,
        capsize=2.6,
        zorder=1,
    )
    ax_right.scatter(phase[x_col], phase["A_kappa_mean"], c=colors, s=42, zorder=2)
    ax_right.plot(phase[x_col], phase["A_kappa_mean"], color="0.45", linewidth=1.1, alpha=0.7, zorder=0)
    for _, row in phase.iterrows():
        ax_right.annotate(
            str(row["condition"]),
            (row[x_col], row["A_kappa_mean"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=7.5,
        )
    if eta_xlim:
        ax_right.set_xlim(-0.03, 0.53)
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title(right_title)
    ax_right.grid(True, alpha=0.24)

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

label_input_name = 'phase_like_A_by_complexity'
manual_input_name = 'phase_like_A_by_complexity'
x_col = 'complexity'
x_label = '3-NN MNIST complexity'
right_title = 'A_kappa by complexity'
eta_xlim = False
label_phase, label_curves, label_inputs = label_phase_frames(label_input_name)
manual_phase, manual_curves, manual_inputs = manual_endpoint_phase_frames(manual_input_name)
phase = pd.concat([manual_phase, label_phase], ignore_index=True).sort_values(x_col)
curves = pd.concat([manual_curves, label_curves], ignore_index=True).sort_values(['eta', 'radius'])
plot_merged_phase(phase, curves, output_png, x_col=x_col, x_label=x_label, right_title=right_title, eta_xlim=eta_xlim)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 63. dnn_mnist/05_proxy_local_entropy/merged/phase_like_A_by_eta.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/merged/phase_like_A_by_eta.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_like_A_by_eta.csv`
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_derivative_curves.csv`
- `03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_like_A_by_rule.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_derivative_curves.csv`
- `03_dnn_mnist/manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/merged/phase_like_A_by_eta.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_like_A_by_eta.csv', ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_derivative_curves.csv', ROOT / '03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_like_A_by_rule.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_derivative_curves.csv', ROOT / '03_dnn_mnist/manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
PLE_INPUT_ROOTS = {
    'label_noise_sweep': DNN_ROOT / 'label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs',
    'manual_rules': DNN_ROOT / 'manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs',
}
ETA_COMPLEXITY_PATH = DNN_ROOT / 'label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'
MANUAL_COMPLEXITY_PATH = DNN_ROOT / 'manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv'
ENDPOINT_ETA = {'real_even_odd': 0.0, 'random_label': 0.5}
ENDPOINT_LABEL = {'real_even_odd': 'even_odd (eta 0.00)', 'random_label': 'random (eta 0.50)'}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def finite_or_zero(values: pd.Series) -> np.ndarray:
    return pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype=float)

def eta_complexity_map() -> dict[float, float]:
    frame = pd.read_csv(require_file(ETA_COMPLEXITY_PATH))
    return {
        round(float(row["eta"]), 10): float(row["complexity_mean"])
        for _, row in frame.dropna(subset=["eta", "complexity_mean"]).iterrows()
    }

def manual_complexity_map() -> dict[str, float]:
    frame = pd.read_csv(require_file(MANUAL_COMPLEXITY_PATH))
    return {
        str(row["rule_name"]): float(row["complexity_mean"])
        for _, row in frame.dropna(subset=["rule_name", "complexity_mean"]).iterrows()
    }

def map_eta_complexity(values: pd.Series) -> pd.Series:
    lookup = eta_complexity_map()
    eta = pd.to_numeric(values, errors="coerce")
    missing = sorted({float(value) for value in eta.dropna() if round(float(value), 10) not in lookup})
    if missing:
        raise ValueError(f"{ETA_COMPLEXITY_PATH} is missing eta values required by PLE phase inputs: {missing}")
    return eta.map(lambda value: lookup[round(float(value), 10)] if pd.notna(value) else np.nan)

def map_manual_complexity(values: pd.Series) -> pd.Series:
    lookup = manual_complexity_map()
    rules = values.astype(str)
    missing = sorted(set(rules.dropna()).difference(lookup))
    if missing:
        raise ValueError(f"{MANUAL_COMPLEXITY_PATH} is missing rules required by PLE phase inputs: {missing}")
    return rules.map(lookup)

def label_phase_frames(name: str) -> tuple[pd.DataFrame, pd.DataFrame, tuple[Path, ...]]:
    phase_path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / f"{name}.csv")
    curves_path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / "phase_derivative_curves.csv")
    phase = pd.read_csv(phase_path)
    curves = pd.read_csv(curves_path)

    phase_out = pd.DataFrame(
        {
            "eta": pd.to_numeric(phase["eta"], errors="coerce"),
            "complexity": map_eta_complexity(phase["eta"]),
            "condition": phase["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "A_kappa_mean": pd.to_numeric(phase["A_kappa_mean"], errors="coerce"),
            "A_kappa_sem": pd.to_numeric(phase["A_kappa_sem"], errors="coerce"),
        }
    )
    curves_out = pd.DataFrame(
        {
            "eta": pd.to_numeric(curves["eta"], errors="coerce"),
            "condition": curves["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "radius": pd.to_numeric(curves["radius"], errors="coerce"),
            "value": pd.to_numeric(curves["dphi_dr_smooth_mean"], errors="coerce"),
            "sem": pd.to_numeric(curves["dphi_dr_smooth_sem"], errors="coerce"),
        }
    )
    return (
        phase_out.dropna(subset=["eta", "A_kappa_mean"]),
        curves_out.dropna(subset=["eta", "radius", "value"]),
        (phase_path, curves_path, ETA_COMPLEXITY_PATH),
    )

def manual_endpoint_phase_frames(name: str) -> tuple[pd.DataFrame, pd.DataFrame, tuple[Path, ...]]:
    phase_path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / f"{name}.csv")
    curves_path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / "phase_derivative_curves.csv")
    phase = pd.read_csv(phase_path)
    curves = pd.read_csv(curves_path)
    phase = phase[phase["rule_name"].isin(ENDPOINT_ETA)].copy()
    curves = curves[curves["rule_name"].isin(ENDPOINT_ETA)].copy()

    phase_out = pd.DataFrame(
        {
            "eta": phase["rule_name"].map(ENDPOINT_ETA),
            "complexity": map_manual_complexity(phase["rule_name"]),
            "condition": phase["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "A_kappa_mean": pd.to_numeric(phase["A_kappa_mean"], errors="coerce"),
            "A_kappa_sem": pd.to_numeric(phase["A_kappa_sem"], errors="coerce"),
        }
    )
    curves_out = pd.DataFrame(
        {
            "eta": curves["rule_name"].map(ENDPOINT_ETA),
            "condition": curves["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "radius": pd.to_numeric(curves["radius"], errors="coerce"),
            "value": pd.to_numeric(curves["dphi_dr_smooth_mean"], errors="coerce"),
            "sem": pd.to_numeric(curves["dphi_dr_smooth_sem"], errors="coerce"),
        }
    )
    return (
        phase_out.dropna(subset=["eta", "A_kappa_mean"]),
        curves_out.dropna(subset=["eta", "radius", "value"]),
        (phase_path, curves_path, MANUAL_COMPLEXITY_PATH),
    )

def plot_merged_phase(
    phase: pd.DataFrame,
    curves: pd.DataFrame,
    path: Path,
    *,
    x_col: str,
    x_label: str,
    right_title: str,
    eta_xlim: bool = False,
) -> None:
    if phase.empty or curves.empty:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.4, 4.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = Normalize(0.0, 0.5)
    handles = []

    for eta, group in curves.groupby("eta", sort=True):
        group = group.sort_values("radius")
        color = cmap(norm(float(eta)))
        source = str(group["source"].iloc[0])
        label = str(group["condition"].iloc[0])
        linestyle = "-" if source == "manual_endpoint" else "--"
        linewidth = 2.1 if source == "manual_endpoint" else 1.55
        x = group["radius"].to_numpy(dtype=float)
        y = group["value"].to_numpy(dtype=float)
        line = ax_left.plot(x, y, color=color, linestyle=linestyle, linewidth=linewidth, label=label)[0]
        handles.append(line)
        err = finite_or_zero(group["sem"])
        ax_left.fill_between(x, y - err, y + err, color=color, alpha=0.10, linewidth=0)

    ax_left.set_xlabel("distance d")
    ax_left.set_ylabel(r"energetic $d\phi/dd$")
    ax_left.set_title("Energetic derivative")
    ax_left.grid(True, alpha=0.24)
    ax_left.legend(handles=handles, frameon=False, fontsize=8.0, ncol=1)
    ax_left.text(
        0.01,
        0.02,
        "solid endpoints; dashed eta sweep",
        transform=ax_left.transAxes,
        fontsize=7.2,
        color="0.35",
    )

    phase = phase.sort_values(x_col)
    colors = [cmap(norm(float(eta))) for eta in phase["eta"]]
    ax_right.errorbar(
        phase[x_col],
        phase["A_kappa_mean"],
        yerr=finite_or_zero(phase["A_kappa_sem"]),
        fmt="none",
        ecolor="0.35",
        elinewidth=1.0,
        capsize=2.6,
        zorder=1,
    )
    ax_right.scatter(phase[x_col], phase["A_kappa_mean"], c=colors, s=42, zorder=2)
    ax_right.plot(phase[x_col], phase["A_kappa_mean"], color="0.45", linewidth=1.1, alpha=0.7, zorder=0)
    for _, row in phase.iterrows():
        ax_right.annotate(
            str(row["condition"]),
            (row[x_col], row["A_kappa_mean"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=7.5,
        )
    if eta_xlim:
        ax_right.set_xlim(-0.03, 0.53)
    ax_right.set_xlabel(x_label)
    ax_right.set_ylabel("A measure")
    ax_right.set_title(right_title)
    ax_right.grid(True, alpha=0.24)

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

label_input_name = 'phase_like_A_by_eta'
manual_input_name = 'phase_like_A_by_rule'
x_col = 'eta'
x_label = 'aligned $\\eta$'
right_title = 'A_kappa by aligned eta'
eta_xlim = True
label_phase, label_curves, label_inputs = label_phase_frames(label_input_name)
manual_phase, manual_curves, manual_inputs = manual_endpoint_phase_frames(manual_input_name)
phase = pd.concat([manual_phase, label_phase], ignore_index=True).sort_values(x_col)
curves = pd.concat([manual_curves, label_curves], ignore_index=True).sort_values(['eta', 'radius'])
plot_merged_phase(phase, curves, output_png, x_col=x_col, x_label=x_label, right_title=right_title, eta_xlim=eta_xlim)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 64. dnn_mnist/05_proxy_local_entropy/merged/phi_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/merged/phi_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/merged/phi_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
PLE_INPUT_ROOTS = {
    'label_noise_sweep': DNN_ROOT / 'label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs',
    'manual_rules': DNN_ROOT / 'manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs',
}
ETA_COMPLEXITY_PATH = DNN_ROOT / 'label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'
MANUAL_COMPLEXITY_PATH = DNN_ROOT / 'manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv'
ENDPOINT_ETA = {'real_even_odd': 0.0, 'random_label': 0.5}
ENDPOINT_LABEL = {'real_even_odd': 'even_odd (eta 0.00)', 'random_label': 'random (eta 0.50)'}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def finite_or_zero(values: pd.Series) -> np.ndarray:
    return pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype=float)

def label_noise_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    out = pd.DataFrame(
        {
            "eta": pd.to_numeric(frame["eta"], errors="coerce"),
            "condition": frame["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def manual_endpoint_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    frame = frame[frame["rule_name"].isin(ENDPOINT_ETA)].copy()
    out = pd.DataFrame(
        {
            "eta": frame["rule_name"].map(ENDPOINT_ETA),
            "condition": frame["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def merged_curve_frame(
    name: str,
    label_value_col: str,
    label_sem_col: str,
    manual_value_col: str,
    manual_sem_col: str,
) -> tuple[pd.DataFrame, tuple[Path, ...]]:
    label_frame, label_path = label_noise_curve_frame(name, label_value_col, label_sem_col)
    manual_frame, manual_path = manual_endpoint_curve_frame(name, manual_value_col, manual_sem_col)
    frame = pd.concat([manual_frame, label_frame], ignore_index=True).sort_values(["eta", "radius"])
    return frame, (label_path, manual_path)

def plot_merged_curves(frame: pd.DataFrame, ylabel: str, title: str, path: Path) -> None:
    if frame.empty:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(8.2, 5.1), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = Normalize(0.0, 0.5)
    handles = []

    for eta, group in frame.groupby("eta", sort=True):
        group = group.sort_values("radius")
        color = cmap(norm(float(eta)))
        source = str(group["source"].iloc[0])
        label = str(group["condition"].iloc[0])
        linestyle = "-" if source == "manual_endpoint" else "--"
        linewidth = 2.1 if source == "manual_endpoint" else 1.65
        x = group["radius"].to_numpy(dtype=float)
        y = group["value"].to_numpy(dtype=float)
        line = ax.plot(x, y, color=color, linestyle=linestyle, linewidth=linewidth, label=label)[0]
        handles.append(line)
        if "sem" in group.columns:
            err = finite_or_zero(group["sem"])
            ax.fill_between(x, y - err, y + err, color=color, alpha=0.11, linewidth=0)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"aligned $\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(handles=handles, frameon=False, fontsize=8.0, ncol=2)
    ax.text(
        0.01,
        0.02,
        "solid: manual endpoints; dashed: label-noise sweep; band: mean +/- SE",
        transform=ax.transAxes,
        fontsize=7.2,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

name = 'phi_d_curve'
label_value_col = 'delta_phi_energy_mean'
label_sem_col = 'delta_phi_energy_sem'
manual_value_col = 'delta_phi_energy_unit_mean'
manual_sem_col = 'delta_phi_energy_unit_sem'
ylabel = '$\\phi(d)-\\phi(d_0)$'
title = 'Merged MNIST phi(d): endpoints + eta sweep'
frame, source_inputs = merged_curve_frame(name, label_value_col, label_sem_col, manual_value_col, manual_sem_col)
plot_merged_curves(frame, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))


## 65. dnn_mnist/05_proxy_local_entropy/merged/phi_energetic_d_curve.png

Source image: `03_dnn_mnist/figures/05_proxy_local_entropy/merged/phi_energetic_d_curve.png`

Inputs:
- `03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv`
- `03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv`

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import math
import re

from IPython.display import Image, display
from matplotlib.colors import Normalize
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    candidates = [Path.cwd() / 'Figures', Path.cwd().parent / 'Figures']
    for candidate in candidates:
        if (candidate / 'manifest.csv').exists():
            FIGURES_DIR = candidate.resolve()
            break
    else:
        raise RuntimeError('Run this notebook from the project root or the Figures directory.')
ROOT = FIGURES_DIR.parent

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

output_png = FIGURES_DIR / 'dnn_mnist/05_proxy_local_entropy/merged/phi_energetic_d_curve.png'
input_paths = [ROOT / '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv', ROOT / '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv']

DNN_ROOT = ROOT / '03_dnn_mnist'
PLE_INPUT_ROOTS = {
    'label_noise_sweep': DNN_ROOT / 'label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs',
    'manual_rules': DNN_ROOT / 'manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs',
}
ETA_COMPLEXITY_PATH = DNN_ROOT / 'label_noise_sweep/02_complexity_measure/summarized_outputs/eta_complexity_summary.csv'
MANUAL_COMPLEXITY_PATH = DNN_ROOT / 'manual_rules/02_complexity_measure/summarized_outputs/manual_rule_complexity_summary.csv'
ENDPOINT_ETA = {'real_even_odd': 0.0, 'random_label': 0.5}
ENDPOINT_LABEL = {'real_even_odd': 'even_odd (eta 0.00)', 'random_label': 'random (eta 0.50)'}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def finite_or_zero(values: pd.Series) -> np.ndarray:
    return pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype=float)

def label_noise_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["label_noise_sweep"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    out = pd.DataFrame(
        {
            "eta": pd.to_numeric(frame["eta"], errors="coerce"),
            "condition": frame["eta"].map(lambda eta: f"eta {float(eta):.2f}"),
            "source": "eta_sweep",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def manual_endpoint_curve_frame(name: str, value_col: str, sem_col: str) -> tuple[pd.DataFrame, Path]:
    path = require_file(PLE_INPUT_ROOTS["manual_rules"] / name / f"{name}.csv")
    frame = pd.read_csv(path)
    frame = frame[frame["rule_name"].isin(ENDPOINT_ETA)].copy()
    out = pd.DataFrame(
        {
            "eta": frame["rule_name"].map(ENDPOINT_ETA),
            "condition": frame["rule_name"].map(ENDPOINT_LABEL),
            "source": "manual_endpoint",
            "radius": pd.to_numeric(frame["radius"], errors="coerce"),
            "value": pd.to_numeric(frame[value_col], errors="coerce"),
            "sem": pd.to_numeric(frame[sem_col], errors="coerce"),
        }
    )
    return out.dropna(subset=["eta", "radius", "value"]), path

def merged_curve_frame(
    name: str,
    label_value_col: str,
    label_sem_col: str,
    manual_value_col: str,
    manual_sem_col: str,
) -> tuple[pd.DataFrame, tuple[Path, ...]]:
    label_frame, label_path = label_noise_curve_frame(name, label_value_col, label_sem_col)
    manual_frame, manual_path = manual_endpoint_curve_frame(name, manual_value_col, manual_sem_col)
    frame = pd.concat([manual_frame, label_frame], ignore_index=True).sort_values(["eta", "radius"])
    return frame, (label_path, manual_path)

def plot_merged_curves(frame: pd.DataFrame, ylabel: str, title: str, path: Path) -> None:
    if frame.empty:
        raise ValueError(f"no finite rows to plot for {path}")

    fig, ax = plt.subplots(figsize=(8.2, 5.1), constrained_layout=True)
    cmap = plt.get_cmap("viridis")
    norm = Normalize(0.0, 0.5)
    handles = []

    for eta, group in frame.groupby("eta", sort=True):
        group = group.sort_values("radius")
        color = cmap(norm(float(eta)))
        source = str(group["source"].iloc[0])
        label = str(group["condition"].iloc[0])
        linestyle = "-" if source == "manual_endpoint" else "--"
        linewidth = 2.1 if source == "manual_endpoint" else 1.65
        x = group["radius"].to_numpy(dtype=float)
        y = group["value"].to_numpy(dtype=float)
        line = ax.plot(x, y, color=color, linestyle=linestyle, linewidth=linewidth, label=label)[0]
        handles.append(line)
        if "sem" in group.columns:
            err = finite_or_zero(group["sem"])
            ax.fill_between(x, y - err, y + err, color=color, alpha=0.11, linewidth=0)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"aligned $\eta$")
    ax.set_xlabel("distance d")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.24)
    ax.legend(handles=handles, frameon=False, fontsize=8.0, ncol=2)
    ax.text(
        0.01,
        0.02,
        "solid: manual endpoints; dashed: label-noise sweep; band: mean +/- SE",
        transform=ax.transAxes,
        fontsize=7.2,
        color="0.35",
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=240)
    plt.close(fig)

name = 'phi_energetic_d_curve'
label_value_col = 'phi_energy_raw_mean'
label_sem_col = 'phi_energy_raw_sem'
manual_value_col = 'phi_energy_raw_mean'
manual_sem_col = 'phi_energy_raw_sem'
ylabel = 'energetic $\\phi(d)$'
title = 'Merged MNIST energetic phi(d): endpoints + eta sweep'
frame, source_inputs = merged_curve_frame(name, label_value_col, label_sem_col, manual_value_col, manual_sem_col)
plot_merged_curves(frame, ylabel, title, output_png)

for path in input_paths:
    print('input:', path.relative_to(ROOT).as_posix())
print('output:', output_png.relative_to(ROOT).as_posix())
display(Image(filename=str(output_png)))
